# Clinical Synthetic Data Generation Framework

This notebook explores the performance of the following Synthetic Table Generation Methods

- **CTGAN** (Conditional Tabular GAN)
- **CTAB-GAN** (Conditional Tabular GAN with advanced preprocessing)
- **CTAB-GAN+** (Enhanced version with WGAN-GP losses, general transforms, and improved stability)
- **GANerAid** (Custom implementation)
- **CopulaGAN** (Copula-based GAN)
- **TVAE** (Variational Autoencoder)

- Section 1 prepares the environment and sources setup.py.
- Section 2 reads in the dataset and produces an initial suite of EDA. 
  - 2.1.1 - user needs to adapt for incoming dataset
  - 2.1.2 - user should review to ensure target colummns and categorical columns are properly identified
  - 2.1.3 - user should confirm that data has loaded properly
  - 2.1.4 - if your data has missing values, MICE is employed; user should review
  - CHUNK_2_1_4_C: This code samples 500 rows to be used in rest of notebook for purposes of testing; comment out this code for production.
  - 2.1.5 - Exploratory data analysis
- Section 3 demonstrates the performance of each metholodogy with some default hyperparameters. 
   - 3.1.1-3.1.6 Subsections for each method
   - 3.2 batch processing of figures/tables from section 3 
- Section 4 runs hyperparameter optimization. 
   - 4.1.1-4.1.6 Subsections for each method
   - 4.2 batch processing of figures/tables from section 4 describing the optimization process 
- Section 5 re-runs each model with their respective best hyperparameters. 
   - 5.1.1-5.1.6 re-run each model using the best hyperparameters identified in Section 4.1.1-4.1.6, resp.
   - 5.2 batch processing of figures/tables from Section 5.


Refer to readme.md, doc\Model-descriptions.md, doc\Objective-function.md.

## 1 Setup and Configuration

In [1]:
# !pip install -r requirements.txt
# !pip install sdv
# !pip install ctgan
# !pip install numpy==1.26.4
# !pip install GANerAid

In [2]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

print("🎯 SETUP MODULE IMPORTED SUCCESSFULLY!")
print("="*60)

[OK] Essential data science libraries imported successfully!
[CONFIG] Session timestamp: 2026-01-07
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
Detected sklearn 1.2.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.2.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementations
[OK] Backward compatibility patches loaded from src/compat.py
[OK] TRTSEvaluator backward compatibility patch applied successfully!
[SETUP] Thin re-export layer loaded successfully!
[SETUP] Session timestamp: 2026-01-07
[SETUP] All functions imported from modular src/ structure
[SETUP] Backward compatibility maintained for notebooks
Session timestamp refreshed to: 2026-01-07
🎯 SETUP MODULE IMPORTED SUCCESSF

## 2 Data Loading and Pre-processing

#### 2.1.1 USER ATTENTION NEEDED

Adapt this for your incoming dataset.

In [3]:
# Code Chunk ID: CHUNK_2_1_1_A
# =================== USER CONFIGURATION ===================
import os
import pandas as pd
import re
import numpy as np
from typing import Dict, List, Tuple, Any
import optuna

# 📝 CONFIGURE YOUR DATASET: Update these settings for your data
DATA_FILE = 'data/Breast_cancer_data.csv'      # Path to your CSV file
TARGET_COLUMN = 'diagnosis'                    # Name of your target/outcome column

# 🔧 DATASET IDENTIFIER (for results folder naming)
# Option 1: Manual override (recommended for consistent naming)
DATASET_IDENTIFIER_OVERRIDE = 'breast-cancer-data'  # Changed to match auto-extraction

# 🔧 OPTIONAL ADVANCED SETTINGS (Auto-detected if left empty)
CATEGORICAL_COLUMNS = []                       # List categorical columns or leave empty for auto-detection - All continuous variables
MISSING_STRATEGY = 'median'                    # Options: 'mice', 'drop', 'median', 'mode'
DATASET_NAME = 'Breast Cancer Dataset'        # Descriptive name for your dataset

# 🚨 IMPORTANT: Verify these settings match your dataset before running!
print(f"📊 Configuration Summary:")
print(f"   Dataset: {DATASET_NAME}")
print(f"   File: {DATA_FILE}")
print(f"   Target: {TARGET_COLUMN}")
print(f"   Manual ID Override: {DATASET_IDENTIFIER_OVERRIDE}")
print(f"   Categorical: {CATEGORICAL_COLUMNS}")
print(f"   Missing Data Strategy: {MISSING_STRATEGY}")

# Load and prepare the dataset
data_file = DATA_FILE
target_column = TARGET_COLUMN

print(f"\n🔍 Loading dataset from: {data_file}")

try:
    data = pd.read_csv(data_file)
    print(f"✅ Dataset loaded successfully!")
    print(f"📊 Original shape: {data.shape}")
    
    # Set up dataset identifier and current data file for new folder structure
    import setup
    if DATASET_IDENTIFIER_OVERRIDE:
        dataset_identifier = DATASET_IDENTIFIER_OVERRIDE
        setup.DATASET_IDENTIFIER = DATASET_IDENTIFIER_OVERRIDE
        setup.CURRENT_DATA_FILE = data_file
        print(f"📁 Using manual dataset identifier: {dataset_identifier}")
    else:
        dataset_identifier = setup.extract_dataset_identifier(data_file)
        setup.DATASET_IDENTIFIER = dataset_identifier
        setup.CURRENT_DATA_FILE = data_file
        print(f"📁 Auto-extracted dataset identifier: {dataset_identifier}")
    
    # 🔧 CRITICAL FIX: Set global DATASET_IDENTIFIER for use in other chunks
    DATASET_IDENTIFIER = dataset_identifier  # This was missing!
    
    # 📁 NEW: Update RESULTS_DIR for organized file outputs using proper structure
    # Don't set a specific RESULTS_DIR here - let each section use get_results_path()
    # This ensures proper date/section structure like: results/dataset/2025-09-12/Section-2/
    RESULTS_DIR = f"results/{dataset_identifier}/"  # Base directory only
    
    print(f"✅ Dataset identifier set: {dataset_identifier}")
    print(f"✅ Global DATASET_IDENTIFIER: {DATASET_IDENTIFIER}")
    print(f"📅 Session timestamp: {setup.SESSION_TIMESTAMP}")
    print(f"🗂️  Results will be saved to: results/{dataset_identifier}/")
    
except FileNotFoundError:
    print(f"❌ Error: File not found at {data_file}")
    print("   Please check the DATA_FILE path in your configuration above")
    print("   Current working directory:", os.getcwd())
    raise

except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    raise

if data is not None:
    print(f"\n📋 Dataset Info:")
    print(f"   • Shape: {data.shape}")
    print(f"   • Columns: {list(data.columns)}")
    
    # Check if target column exists
    if target_column not in data.columns:
        print(f"\n❌ WARNING: Target column '{target_column}' not found!")
        print(f"   Available columns: {list(data.columns)}")
        print("   Please update TARGET_COLUMN in the configuration above")
    else:
        print(f"   • Target column '{target_column}' found ✅")
        print(f"   • Target distribution: {data[target_column].value_counts().to_dict()}")
    
    # Check for missing values
    missing_values = data.isnull().sum()
    if missing_values.sum() > 0:
        print(f"\n⚠️  Missing values detected:")
        for col, count in missing_values[missing_values > 0].items():
            print(f"   • {col}: {count} missing ({count/len(data)*100:.1f}%)")
    else:
        print(f"\n✅ No missing values detected")
else:
    print("\n❌ Dataset loading failed - please fix the configuration and try again")

📊 Configuration Summary:
   Dataset: Breast Cancer Dataset
   File: data/Breast_cancer_data.csv
   Target: diagnosis
   Manual ID Override: breast-cancer-data
   Categorical: []
   Missing Data Strategy: median

🔍 Loading dataset from: data/Breast_cancer_data.csv
✅ Dataset loaded successfully!
📊 Original shape: (569, 6)
📁 Using manual dataset identifier: breast-cancer-data
✅ Dataset identifier set: breast-cancer-data
✅ Global DATASET_IDENTIFIER: breast-cancer-data
📅 Session timestamp: 2026-01-07
🗂️  Results will be saved to: results/breast-cancer-data/

📋 Dataset Info:
   • Shape: (569, 6)
   • Columns: ['mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'diagnosis']
   • Target column 'diagnosis' found ✅
   • Target distribution: {1: 357, 0: 212}

✅ No missing values detected


The code defines utilities for column name standardization and dataset analysis using the pandas library in Python. It includes functions to clean and normalize column names, identify the target variable, categorize column types, and validate dataset configurations. These functions enhance data preprocessing by ensuring consistency and integrity, making it easier to manage various data types and handle potential issues like missing values. Overall, they provide a structured approach for effective dataset analysis.

#### 2.1.2 Column Name Standardization and Dataset Analysis Utilities

In [4]:
# Code Chunk ID: CHUNK_2_1_2_A
# Column Name Standardization and Dataset Analysis Utilities
import re
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Any

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Create mapping of old to new column names
    name_mapping = {}
    
    for col in df.columns:
        # Remove special characters and normalize
        new_name = re.sub(r'[^\w\s]', '', str(col))  # Remove special chars
        new_name = re.sub(r'\s+', '_', new_name.strip())  # Replace spaces with underscores
        new_name = new_name.lower()  # Convert to lowercase
        
        # Handle duplicate names
        if new_name in name_mapping.values():
            counter = 1
            while f"{new_name}_{counter}" in name_mapping.values():
                counter += 1
            new_name = f"{new_name}_{counter}"
            
        name_mapping[col] = new_name
    
    # Rename columns
    df = df.rename(columns=name_mapping)
    
    print(f"🔄 Column Name Standardization:")
    for old, new in name_mapping.items():
        if old != new:
            print(f"   '{old}' → '{new}'")
    
    return df, name_mapping

def detect_target_column(df: pd.DataFrame, target_hint: str = None) -> str:
    """
    Detect the target column in the dataset.
    
    Args:
        df: Input dataframe
        target_hint: User-provided hint for target column name
        
    Returns:
        Name of the detected target column
    """
    # Common target column patterns
    target_patterns = [
        'target', 'label', 'class', 'outcome', 'result', 'diagnosis', 
        'response', 'y', 'dependent', 'output', 'prediction'
    ]
    
    # If user provided hint, try to find it first
    if target_hint:
        # Try exact match (case insensitive)
        for col in df.columns:
            if col.lower() == target_hint.lower():
                print(f"✅ Target column found: '{col}' (user specified)")
                return col
        
        # Try partial match
        for col in df.columns:
            if target_hint.lower() in col.lower():
                print(f"✅ Target column found: '{col}' (partial match to '{target_hint}')")
                return col
    
    # Auto-detect based on patterns
    for pattern in target_patterns:
        for col in df.columns:
            if pattern in col.lower():
                print(f"✅ Target column auto-detected: '{col}' (pattern: '{pattern}')")
                return col
    
    # If no pattern match, check for binary columns (likely targets)
    binary_cols = []
    for col in df.columns:
        unique_vals = df[col].dropna().nunique()
        if unique_vals == 2:
            binary_cols.append(col)
    
    if binary_cols:
        target_col = binary_cols[0]  # Take first binary column
        print(f"✅ Target column inferred: '{target_col}' (binary column)")
        return target_col
    
    # Last resort: use last column
    target_col = df.columns[-1]
    print(f"⚠️ Target column defaulted to: '{target_col}' (last column)")
    return target_col

def analyze_column_types(df: pd.DataFrame, categorical_hint: List[str] = None) -> Dict[str, str]:
    """
    Analyze and categorize column types.
    
    Args:
        df: Input dataframe
        categorical_hint: User-provided list of categorical columns
        
    Returns:
        Dictionary mapping column names to types ('categorical', 'continuous', 'binary')
    """
    column_types = {}
    
    for col in df.columns:
        # Skip if user explicitly specified as categorical
        if categorical_hint and col in categorical_hint:
            column_types[col] = 'categorical'
            continue
            
        # Analyze column characteristics
        non_null_data = df[col].dropna()
        unique_count = non_null_data.nunique()
        total_count = len(non_null_data)
        
        # Determine type based on data characteristics
        if unique_count == 2:
            column_types[col] = 'binary'
        elif df[col].dtype == 'object' or unique_count < 10:
            column_types[col] = 'categorical'
        elif df[col].dtype in ['int64', 'float64'] and unique_count > 10:
            column_types[col] = 'continuous'
        else:
            # Default based on uniqueness ratio
            uniqueness_ratio = unique_count / total_count
            if uniqueness_ratio < 0.1:
                column_types[col] = 'categorical'
            else:
                column_types[col] = 'continuous'
    
    return column_types

def validate_dataset_config(df: pd.DataFrame, target_col: str, config: Dict[str, Any]) -> bool:
    """
    Validate dataset configuration and provide warnings.
    
    Args:
        df: Input dataframe
        target_col: Target column name
        config: Configuration dictionary
        
    Returns:
        True if validation passes, False otherwise
    """
    print(f"\n🔍 Dataset Validation:")
    
    valid = True
    
    # Check if target column exists
    if target_col not in df.columns:
        print(f"❌ Target column '{target_col}' not found in dataset!")
        print(f"   Available columns: {list(df.columns)}")
        valid = False
    else:
        print(f"✅ Target column '{target_col}' found")
    
    # Check dataset size
    if len(df) < 100:
        print(f"⚠️ Small dataset: {len(df)} rows (recommend >1000 for synthetic data)")
    else:
        print(f"✅ Dataset size: {len(df)} rows")
    
    # Check for missing data
    missing_pct = (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
    if missing_pct > 20:
        print(f"⚠️ High missing data: {missing_pct:.1f}% (recommend MICE imputation)")
    elif missing_pct > 0:
        print(f"🔍 Missing data: {missing_pct:.1f}% (manageable)")
    else:
        print(f"✅ No missing data")
    
    return valid

print("✅ Dataset analysis utilities loaded successfully!")

✅ Dataset analysis utilities loaded successfully!


#### 2.1.3 Load and Analyze Dataset with Generalized Configuration

This code loads and analyzes a dataset using a specified configuration. It imports necessary libraries, attempts to read a CSV file, and standardizes the column names while allowing for potential updates to the target column. The script detects the target column, analyzes data types, and validates the dataset configuration, providing a summary of the dataset’s shape and missing values. Finally, it stores metadata about the dataset for future reference.

In [5]:
# Code Chunk ID: CHUNK_2_1_3_A
# Load and Analyze Dataset with Generalized Configuration
import pandas as pd
import numpy as np

# Apply user configuration
data_file = DATA_FILE
target_column = TARGET_COLUMN

print(f"📂 Loading dataset: {data_file}")

# Load the dataset
try:
    data = pd.read_csv(data_file)
    print(f"✅ Dataset loaded successfully!")
    print(f"📊 Original shape: {data.shape}")
    
    # Set up dataset identifier and current data file for new folder structure
    import setup
    setup.DATASET_IDENTIFIER = setup.extract_dataset_identifier(data_file)
    setup.CURRENT_DATA_FILE = data_file
    print(f"📁 Dataset identifier: {setup.DATASET_IDENTIFIER}")
    print(f"📅 Session timestamp: {setup.SESSION_TIMESTAMP}")
    
except FileNotFoundError:
    print(f"❌ Error: Could not find file {data_file}")
    print(f"📋 Please verify the file path in the USER CONFIGURATION section above")
    raise
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    raise

# Basic info
print(f"\n📋 Dataset Info:")
print(f"   • Target column: {target_column}")
print(f"   • Features: {data.shape[1] - 1}")
print(f"   • Samples: {data.shape[0]}")
print(f"   • Memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

📂 Loading dataset: data/Breast_cancer_data.csv
✅ Dataset loaded successfully!
📊 Original shape: (569, 6)
📁 Dataset identifier: breast-cancer-data
📅 Session timestamp: 2026-01-07

📋 Dataset Info:
   • Target column: diagnosis
   • Features: 5
   • Samples: 569
   • Memory usage: 0.03 MB


This code provides advanced utilities for handling missing data using various strategies in Python. It includes functions to assess missing data patterns, apply Multiple Imputation by Chained Equations (MICE), visualize missing patterns, and implement different strategies for managing missing values. The `assess_missing_patterns` function analyzes and summarizes missing data, while `apply_mice_imputation` leverages an iterative imputer for numeric columns. The `visualize_missing_patterns` function creates visual representations of missing data, and the `handle_missing_data_strategy` function executes the chosen strategy, offering options like MICE, dropping rows, or filling with median or mode values. Collectively, these utilities facilitate effective management of missing data to improve dataset quality.

#### 2.1.4 Advanced Missing Data Handling with MICE

This code provides a comprehensive toolkit for analyzing, visualizing, and handling missing data in a pandas DataFrame using advanced and flexible approaches. It includes functions to assess the extent and patterns of missingness, visualize those patterns, and apply various imputation techniques. The centerpiece is the ability to perform Multiple Imputation by Chained Equations (MICE) using scikit-learn’s IterativeImputer with a RandomForestRegressor (for numerical features), which statistically estimates and fills in missing values based on all other feature relationships. For categorical variables, the code defaults to simpler mode imputation. Users can also choose alternative strategies like dropping rows (drop), filling with column medians (median), or filling with the most frequent value (mode). The code features detailed feedback and visual support with heatmaps and bar plots to help the user understand and monitor missing data patterns throughout the handling process. Together, these utilities help users robustly prepare data for machine learning or statistical analysis, reducing bias and maximizing data retention and utility.

In [6]:
# Code Chunk ID: CHUNK_2_1_4_A
# Advanced Missing Data Handling with MICE
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def assess_missing_patterns(df: pd.DataFrame) -> dict:
    """
    Comprehensive assessment of missing data patterns.
    
    Args:
        df: Input dataframe
        
    Returns:
        Dictionary with missing data analysis
    """
    missing_analysis = {}
    
    # Basic missing statistics
    missing_counts = df.isnull().sum()
    missing_percentages = (missing_counts / len(df)) * 100
    
    missing_analysis['missing_counts'] = missing_counts[missing_counts > 0]
    missing_analysis['missing_percentages'] = missing_percentages[missing_percentages > 0]
    missing_analysis['total_missing_cells'] = df.isnull().sum().sum()
    missing_analysis['total_cells'] = df.size
    missing_analysis['overall_missing_rate'] = (missing_analysis['total_missing_cells'] / missing_analysis['total_cells']) * 100
    
    # Missing patterns
    missing_patterns = df.isnull().value_counts()
    missing_analysis['missing_patterns'] = missing_patterns
    
    return missing_analysis

def apply_mice_imputation(df: pd.DataFrame, target_col: str = None, max_iter: int = 10, random_state: int = 42) -> pd.DataFrame:
    """
    Apply Multiple Imputation by Chained Equations (MICE) to handle missing data.
    
    Args:
        df: Input dataframe with missing values
        target_col: Target column name (excluded from imputation predictors)
        max_iter: Maximum number of imputation iterations
        random_state: Random state for reproducibility
        
    Returns:
        DataFrame with imputed values
    """
    print(f"🔧 Applying MICE imputation...")
    
    # Separate features and target
    if target_col and target_col in df.columns:
        features = df.drop(columns=[target_col])
        target = df[target_col]
    else:
        features = df.copy()
        target = None
    
    # Identify numeric and categorical columns
    numeric_cols = features.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = features.select_dtypes(include=['object', 'category']).columns.tolist()
    
    df_imputed = features.copy()
    
    # Handle numeric columns with MICE
    if numeric_cols:
        print(f"   Imputing {len(numeric_cols)} numeric columns...")
        numeric_imputer = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=10, random_state=random_state),
            max_iter=max_iter,
            random_state=random_state
        )
        
        numeric_imputed = numeric_imputer.fit_transform(features[numeric_cols])
        df_imputed[numeric_cols] = numeric_imputed
    
    # Handle categorical columns with mode imputation (simpler approach)
    if categorical_cols:
        print(f"   Imputing {len(categorical_cols)} categorical columns with mode...")
        for col in categorical_cols:
            mode_value = features[col].mode()
            if len(mode_value) > 0:
                df_imputed[col] = features[col].fillna(mode_value[0])
            else:
                # If no mode, fill with 'Unknown'
                df_imputed[col] = features[col].fillna('Unknown')
    
    # Add target back if it exists
    if target is not None:
        df_imputed[target_col] = target
    
    print(f"✅ MICE imputation completed!")
    print(f"   Missing values before: {features.isnull().sum().sum()}")
    print(f"   Missing values after: {df_imputed.isnull().sum().sum()}")
    
    return df_imputed

def visualize_missing_patterns(df: pd.DataFrame, title: str = "Missing Data Patterns") -> None:
    """
    Create visualizations for missing data patterns.
    
    Args:
        df: Input dataframe
        title: Title for the plot
    """
    missing_data = df.isnull()
    
    if missing_data.sum().sum() == 0:
        print("✅ No missing data to visualize!")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Missing data heatmap
    sns.heatmap(missing_data, 
                yticklabels=False, 
                cbar=True, 
                cmap='viridis',
                ax=axes[0])
    axes[0].set_title('Missing Data Heatmap')
    axes[0].set_xlabel('Columns')
    
    # Missing data bar chart
    missing_counts = missing_data.sum()
    missing_counts = missing_counts[missing_counts > 0]
    
    if len(missing_counts) > 0:
        missing_counts.plot(kind='bar', ax=axes[1], color='coral')
        axes[1].set_title('Missing Values by Column')
        axes[1].set_ylabel('Count of Missing Values')
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].text(0.5, 0.5, 'No Missing Data', 
                    horizontalalignment='center', 
                    verticalalignment='center',
                    transform=axes[1].transAxes,
                    fontsize=16)
        axes[1].set_title('Missing Values by Column')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

def handle_missing_data_strategy(df: pd.DataFrame, strategy: str, target_col: str = None) -> pd.DataFrame:
    """
    Apply the specified missing data handling strategy.
    
    Args:
        df: Input dataframe
        strategy: Strategy to use ('mice', 'drop', 'median', 'mode')
        target_col: Target column name
        
    Returns:
        DataFrame with missing data handled
    """
    print(f"\n🔧 Applying missing data strategy: {strategy.upper()}")
    
    if df.isnull().sum().sum() == 0:
        print("✅ No missing data detected - no imputation needed")
        return df.copy()
    
    if strategy.lower() == 'mice':
        return apply_mice_imputation(df, target_col)
    
    elif strategy.lower() == 'drop':
        print(f"   Dropping rows with missing values...")
        df_clean = df.dropna()
        print(f"   Rows before: {len(df)}, Rows after: {len(df_clean)}")
        return df_clean
    
    elif strategy.lower() == 'median':
        print(f"   Filling missing values with median/mode...")
        df_filled = df.copy()
        
        # Numeric columns: fill with median
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if df[col].isnull().sum() > 0:
                median_val = df[col].median()
                df_filled[col] = df[col].fillna(median_val)
                print(f"     {col}: filled {df[col].isnull().sum()} values with median {median_val:.2f}")
        
        # Categorical columns: fill with mode
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns
        for col in categorical_cols:
            if df[col].isnull().sum() > 0:
                mode_val = df[col].mode()
                if len(mode_val) > 0:
                    df_filled[col] = df[col].fillna(mode_val[0])
                    print(f"     {col}: filled {df[col].isnull().sum()} values with mode '{mode_val[0]}'")
        
        return df_filled
    
    elif strategy.lower() == 'mode':
        print(f"   Filling missing values with mode...")
        df_filled = df.copy()
        
        for col in df.columns:
            if df[col].isnull().sum() > 0:
                mode_val = df[col].mode()
                if len(mode_val) > 0:
                    df_filled[col] = df[col].fillna(mode_val[0])
                    print(f"     {col}: filled {df[col].isnull().sum()} values with mode '{mode_val[0]}'")
        
        return df_filled
    
    else:
        print(f"⚠️ Unknown strategy '{strategy}'. Using 'median' as fallback.")
        return handle_missing_data_strategy(df, 'median', target_col)

print("✅ Missing data handling utilities loaded successfully!")

✅ Missing data handling utilities loaded successfully!


In [7]:
# Code Chunk ID: CHUNK_2_1_4_B
# ============================================================================
# CONDITIONAL MISSING DATA IMPUTATION
# ============================================================================
# Apply missing data strategy only if missing values exist

missing_count = data.isnull().sum().sum()

if missing_count > 0:
    print(f"🔧 MISSING DATA IMPUTATION")
    print(f"📊 Found {missing_count} missing values - applying {MISSING_STRATEGY} strategy")
    
    # Store original data
    data_original = data.copy()
    
    # Apply imputation using CHUNK_008 functions
    data = handle_missing_data_strategy(data, MISSING_STRATEGY, TARGET_COLUMN)
    
    # Report results
    remaining = data.isnull().sum().sum()
    print(f"✅ Imputation complete: {missing_count} → {remaining} missing values")
else:
    print("✅ No missing values detected - skipping imputation")

✅ No missing values detected - skipping imputation


⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️

Note that next chunk samples 500 rows

⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️

In [8]:
# CHUNK_2_1_4_C
# This chunk samples 500 rows for quicker testing - remove for full dataset
data=data.sample(n=500, random_state=42)
# data.shape
# data.head()

#### 2.1.5 EDA
This code snippet provides an enhanced overview and analysis of a dataset. It generates basic statistics, including the dataset name, shape, memory usage, total missing values, missing percentage, number of duplicate rows, and counts of numeric and categorical columns. The results are organized into a dictionary called `overview_stats`, which is then iterated over to print each statistic in a formatted manner. Additionally, it sets up for displaying a sample of the data afterward.

In [9]:
# Code Chunk ID: CHUNK_2_1_5_A
# Enhanced Dataset Overview and Analysis
print("📋 COMPREHENSIVE DATASET OVERVIEW")
print("=" * 60)

# Basic statistics
overview_stats = {
    'Dataset Name': 'Breast Cancer Wisconsin (Diagnostic)',
    'Shape': f"{data.shape[0]} rows × {data.shape[1]} columns",
    'Memory Usage': f"{data.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
    'Total Missing Values': data.isnull().sum().sum(),
    'Missing Percentage': f"{(data.isnull().sum().sum() / data.size) * 100:.2f}%",
    'Duplicate Rows': data.duplicated().sum(),
    'Numeric Columns': len(data.select_dtypes(include=[np.number]).columns),
    'Categorical Columns': len(data.select_dtypes(include=['object']).columns)
}

for key, value in overview_stats.items():
    print(f"{key:.<25} {value}")

📋 COMPREHENSIVE DATASET OVERVIEW
Dataset Name............. Breast Cancer Wisconsin (Diagnostic)
Shape.................... 500 rows × 6 columns
Memory Usage............. 0.03 MB
Total Missing Values..... 0
Missing Percentage....... 0.00%
Duplicate Rows........... 0
Numeric Columns.......... 6
Categorical Columns...... 0


In [10]:
# Code Chunk ID: CHUNK_2_1_5_B
# Enhanced Column Analysis - OUTPUT TO FILE
print("📊 DETAILED COLUMN ANALYSIS (SAVING TO FILE)")
print("=" * 50)

column_analysis = pd.DataFrame({
    'Column': data.columns,
    'Data_Type': data.dtypes.astype(str),
    'Unique_Values': [data[col].nunique() for col in data.columns],
    'Missing_Count': [data[col].isnull().sum() for col in data.columns],
    'Missing_Percent': [f"{(data[col].isnull().sum()/len(data)*100):.2f}%" for col in data.columns],
    'Min_Value': [data[col].min() if data[col].dtype in ['int64', 'float64'] else 'N/A' for col in data.columns],
    'Max_Value': [data[col].max() if data[col].dtype in ['int64', 'float64'] else 'N/A' for col in data.columns]
})

# Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
results_path = get_results_path(DATASET_IDENTIFIER, 2)
os.makedirs(results_path, exist_ok=True)
csv_file = f'{results_path}/column_analysis.csv'
column_analysis.to_csv(csv_file, index=False)

print(f"📊 Column analysis table saved to {csv_file}")
print(f"📊 Analysis completed for {len(data.columns)} features")

📊 DETAILED COLUMN ANALYSIS (SAVING TO FILE)
📊 Column analysis table saved to results/breast-cancer-data/2026-01-07/Section-2/column_analysis.csv
📊 Analysis completed for 6 features


This code conducts an enhanced analysis of the target variable within a dataset. It computes the counts and percentages of target classes, organizing the results into a DataFrame called `target_summary`, which distinguishes between benign and malignant classes if applicable. The class balance is assessed by calculating a balance ratio, with outputs indicating whether the dataset is balanced, moderately imbalanced, or highly imbalanced. If the specified target column is not found, it displays a warning and lists available columns in the dataset.

In [11]:
# Code Chunk ID: CHUNK_2_1_5_C
# Enhanced Target Variable Analysis - OUTPUT TO FILE
print("🎯 TARGET VARIABLE ANALYSIS (SAVING TO FILE)")
print("=" * 40)

if target_column in data.columns:
    target_counts = data[target_column].value_counts().sort_index()
    target_props = data[target_column].value_counts(normalize=True).sort_index() * 100
    
    target_summary = pd.DataFrame({
        'Class': target_counts.index,
        'Count': target_counts.values,
        'Percentage': [f"{prop:.1f}%" for prop in target_props.values],
        'Description': ['Benign (Non-cancerous)', 'Malignant (Cancerous)'] if len(target_counts) == 2 else [f'Class {i}' for i in target_counts.index]
    })
    
    # Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
    results_path = get_results_path(DATASET_IDENTIFIER, 2)
    os.makedirs(results_path, exist_ok=True)
    csv_file = f'{results_path}/target_analysis.csv'
    target_summary.to_csv(csv_file, index=False)
    
    # Calculate class balance metrics
    balance_ratio = target_counts.min() / target_counts.max()
    
    # Save balance metrics to separate file
    balance_metrics = pd.DataFrame({
        'Metric': ['Class_Balance_Ratio', 'Dataset_Balance_Category'],
        'Value': [f"{balance_ratio:.3f}", 
                 'Balanced' if balance_ratio > 0.8 else 'Moderately Imbalanced' if balance_ratio > 0.5 else 'Highly Imbalanced']
    })
    balance_file = f'{results_path}/target_balance_metrics.csv'
    balance_metrics.to_csv(balance_file, index=False)
    
    print(f"📊 Target variable analysis saved to {csv_file}")
    print(f"📊 Class balance metrics saved to {balance_file}")
    print(f"📊 Class Balance Ratio: {balance_ratio:.3f}")
    print(f"📊 Dataset Balance: {'Balanced' if balance_ratio > 0.8 else 'Moderately Imbalanced' if balance_ratio > 0.5 else 'Highly Imbalanced'}")
    
else:
    print(f"⚠️ Warning: Target column '{target_column}' not found!")
    print(f"Available columns: {list(data.columns)}")

🎯 TARGET VARIABLE ANALYSIS (SAVING TO FILE)
📊 Target variable analysis saved to results/breast-cancer-data/2026-01-07/Section-2/target_analysis.csv
📊 Class balance metrics saved to results/breast-cancer-data/2026-01-07/Section-2/target_balance_metrics.csv
📊 Class Balance Ratio: 0.597
📊 Dataset Balance: Moderately Imbalanced


This code provides enhanced visualizations of feature distributions in a dataset. It retrieves numeric columns, excluding the target variable, and generates histograms for each numeric feature, displaying them in a grid layout. The histograms are enhanced with options for density, color, and grid lines to improve readability. If no numeric features are found, a warning message is displayed; otherwise, the generated plots give insights into the distributions of the numeric features in the dataset.

In [12]:
# Code Chunk ID: CHUNK_2_1_5_D
# Enhanced Feature Distribution Visualizations - OUTPUT TO FILE
print("📊 FEATURE DISTRIBUTION ANALYSIS (SAVING TO FILE)")
print("=" * 40)

# Turn off interactive mode to prevent figures from displaying in notebook
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
plt.ioff()  # Turn off interactive mode

# Get numeric columns excluding target
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if target_column in numeric_cols:
    numeric_cols.remove(target_column)

if numeric_cols:
    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    # Use dataset name fallback for title
    dataset_name = DATASET_IDENTIFIER.title() if DATASET_IDENTIFIER else "Dataset"
    fig.suptitle(f'{dataset_name} - Feature Distributions', fontsize=16, fontweight='bold')
    
    # Handle different subplot configurations
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        if i < len(axes):
            # Enhanced histogram
            axes[i].hist(data[col], bins=30, alpha=0.7, color='skyblue', 
                        edgecolor='black', density=True)
            
            axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Density')
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for j in range(len(numeric_cols), len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    
    # Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
    results_path = get_results_path(DATASET_IDENTIFIER, 2)
    os.makedirs(results_path, exist_ok=True)
    plot_file = f'{results_path}/feature_distributions.png'
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    plt.close()  # Close the figure to free memory
    
    print(f"📊 Feature distribution plots saved to {plot_file}")
    print(f"📊 Distribution analysis completed for {len(numeric_cols)} numeric features")
else:
    print("⚠️ No numeric features found for visualization")

📊 FEATURE DISTRIBUTION ANALYSIS (SAVING TO FILE)
📊 Feature distribution plots saved to results/breast-cancer-data/2026-01-07/Section-2/feature_distributions.png
📊 Distribution analysis completed for 5 numeric features


This code conducts an enhanced correlation analysis of features within a dataset. It calculates the correlation matrix for numeric columns and includes the target variable if it is numeric, displaying the results in a heatmap for better visualization. The analysis identifies correlations with the target variable, categorizing each feature based on its correlation strength (strong, moderate, or weak) and presenting the findings in a DataFrame. If there are insufficient numeric features, a warning message is displayed, indicating that correlation analysis cannot be performed.

In [13]:
# Code Chunk ID: CHUNK_2_1_5_E
# Enhanced Correlation Analysis - OUTPUT TO FILE
print("🔍 CORRELATION ANALYSIS (SAVING TO FILE)")
print("=" * 30)

# Turn off interactive mode to prevent figures from displaying in notebook
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
plt.ioff()  # Turn off interactive mode

if len(numeric_cols) > 1:
    # Include target in correlation if numeric
    cols_for_corr = numeric_cols.copy()
    if data[target_column].dtype in ['int64', 'float64']:
        cols_for_corr.append(target_column)
    
    correlation_matrix = data[cols_for_corr].corr()
    
    # Enhanced correlation heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(correlation_matrix, 
                annot=True, 
                cmap='RdBu_r',
                center=0, 
                square=True, 
                linewidths=0.5,
                fmt='.3f',
                ax=ax)
    
    # Use dataset name fallback for title
    dataset_name = DATASET_IDENTIFIER.title() if DATASET_IDENTIFIER else "Dataset"
    ax.set_title(f'{dataset_name} - Feature Correlation Matrix', 
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    
    # Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
    results_path = get_results_path(DATASET_IDENTIFIER, 2)
    os.makedirs(results_path, exist_ok=True)
    heatmap_file = f'{results_path}/correlation_heatmap.png'
    plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
    plt.close()  # Close the figure to free memory
    
    # Save correlation matrix to CSV
    corr_matrix_file = f'{results_path}/correlation_matrix.csv'
    correlation_matrix.to_csv(corr_matrix_file)
    
    print(f"🔍 Correlation heatmap saved to {heatmap_file}")
    print(f"🔍 Correlation matrix saved to {corr_matrix_file}")
    
    # Correlation with target analysis
    if target_column in correlation_matrix.columns:
        print("\n🔍 CORRELATIONS WITH TARGET VARIABLE (SAVING TO FILE)")
        print("=" * 45)
        
        target_corrs = correlation_matrix[target_column].abs().sort_values(ascending=False)
        target_corrs = target_corrs[target_corrs.index != target_column]
        
        corr_analysis = pd.DataFrame({
            'Feature': target_corrs.index,
            'Absolute_Correlation': target_corrs.values,
            'Raw_Correlation': [correlation_matrix.loc[feat, target_column] for feat in target_corrs.index],
            'Strength': ['Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.3 else 'Weak' 
                        for corr in target_corrs.values]
        })
        
        # Save correlation analysis to CSV instead of displaying
        corr_analysis_file = f'{results_path}/target_correlations.csv'
        corr_analysis.to_csv(corr_analysis_file, index=False)
        
        print(f"🔍 Target correlation analysis saved to {corr_analysis_file}")
        print(f"📊 Correlation analysis completed for {len(target_corrs)} features")
    
else:
    print("⚠️ Insufficient numeric features for correlation analysis")

🔍 CORRELATION ANALYSIS (SAVING TO FILE)
🔍 Correlation heatmap saved to results/breast-cancer-data/2026-01-07/Section-2/correlation_heatmap.png
🔍 Correlation matrix saved to results/breast-cancer-data/2026-01-07/Section-2/correlation_matrix.csv

🔍 CORRELATIONS WITH TARGET VARIABLE (SAVING TO FILE)
🔍 Target correlation analysis saved to results/breast-cancer-data/2026-01-07/Section-2/target_correlations.csv
📊 Correlation analysis completed for 5 features


This code sets up global configuration variables for consistent evaluation across model evaluations. It checks for the existence of required variables, such as `data` and `target_column`, and raises an error if they are not defined. The code establishes global constants for the target column, results directory, and a copy of the original data while defining categorical columns, excluding the target. It then creates the results directory if it does not already exist and verifies that all necessary global variables are present, providing feedback on the setup's success.

In [14]:
# Code Chunk ID: CHUNK_2_1_5_F
# ============================================================================
# GLOBAL CONFIGURATION VARIABLES
# ============================================================================
# These variables are used across all sections for consistent evaluation

# Verify required variables exist before setting globals
if 'data' not in globals() or 'target_column' not in globals():
    raise ValueError("❌ ERROR: 'data' and 'target_column' must be defined before setting global variables. Please run the data loading cell first.")

# Set up global variables for use in all model evaluations
TARGET_COLUMN = target_column  # Use the target column from data loading

# 🔧 UPDATED: Preserve dataset-specific RESULTS_DIR that was set in CHUNK_005
# Don't override it with a generic path - maintain the structured approach
if 'RESULTS_DIR' not in globals() or RESULTS_DIR is None:
    # Fallback: reconstruct proper results directory structure
    RESULTS_DIR = f"results/{setup.DATASET_IDENTIFIER}/"
    print(f"⚠️  RESULTS_DIR was missing - using fallback: {RESULTS_DIR}")
else:
    print(f"✅ Using existing RESULTS_DIR: {RESULTS_DIR}")

original_data = data.copy()    # Create a copy of original data for evaluation functions

# Define categorical columns for all models
categorical_columns = data.select_dtypes(include=['object']).columns.tolist()
if TARGET_COLUMN in categorical_columns:
    categorical_columns.remove(TARGET_COLUMN)  # Remove target from categorical list

# Apply user-specified categorical columns if provided
if 'CATEGORICAL_COLUMNS' in globals() and CATEGORICAL_COLUMNS:
    categorical_columns = CATEGORICAL_COLUMNS
    print(f"   • Using user-specified categorical columns: {categorical_columns}")
else:
    print(f"   • Auto-detected categorical columns: {categorical_columns}")

print("🔧 Global Configuration Summary:")
print(f"   • TARGET_COLUMN: {TARGET_COLUMN}")
print(f"   • RESULTS_DIR: {RESULTS_DIR}")
print(f"   • original_data shape: {original_data.shape}")
print(f"   • categorical_columns: {categorical_columns}")

# Create base results directory if it doesn't exist
import os
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"   • Created base results directory: {RESULTS_DIR}")
else:
    print(f"   • Base results directory already exists: {RESULTS_DIR}")

# Validate that all required variables are now available
required_vars = ['TARGET_COLUMN', 'RESULTS_DIR', 'original_data', 'categorical_columns']
missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    raise ValueError(f"❌ ERROR: Missing required global variables: {missing_vars}")
else:
    print("✅ All required global variables are now available for Section 3 evaluations")

✅ Using existing RESULTS_DIR: results/breast-cancer-data/
   • Auto-detected categorical columns: []
🔧 Global Configuration Summary:
   • TARGET_COLUMN: diagnosis
   • RESULTS_DIR: results/breast-cancer-data/
   • original_data shape: (500, 6)
   • categorical_columns: []
   • Base results directory already exists: results/breast-cancer-data/
✅ All required global variables are now available for Section 3 evaluations


## 3 Demo All Models with Default Parameters

### 3.1 Demos

Each model is run with default parameters for purposes of testing.

#### 3.1.1 CTGAN Demo

In [15]:
# Code Chunk ID: CHUNK_3_1_1_A
import time
try:
    print("🔄 CTGAN Demo - Default Parameters")
    print("=" * 500)
    
    # Import and initialize CTGAN model using ModelFactory
    from src.models.model_factory import ModelFactory
    
    ctgan_model = ModelFactory.create("ctgan", random_state=42)
    
    # Define demo parameters for quick execution
    demo_params = {
        'epochs': 500,
        'batch_size': 100,
        'generator_dim': (128, 128),
        'discriminator_dim': (128, 128)
    }
    
    # Train with demo parameters
    print("Training CTGAN with demo parameters...")
    start_time = time.time()
    
    # Auto-detect discrete columns
    discrete_columns = data.select_dtypes(include=['object']).columns.tolist()
    
    ctgan_model.train(data, discrete_columns=discrete_columns, **demo_params)
    train_time = time.time() - start_time
    
    # Generate synthetic data
    demo_samples = len(data)  # Same size as original dataset
    print(f"Generating {demo_samples} synthetic samples...")
    synthetic_data_ctgan = ctgan_model.generate(demo_samples)
    
    print(f"✅ CTGAN Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ctgan)}")
    print(f"   - Original data shape: {data.shape}")
    print(f"   - Synthetic data shape: {synthetic_data_ctgan.shape}")
    
    # Store for later use in comprehensive evaluation
    demo_results_ctgan = {
        'model': ctgan_model,
        'synthetic_data': synthetic_data_ctgan,
        'training_time': train_time,
        'parameters_used': demo_params
    }
    
except ImportError as e:
    print(f"❌ CTGAN not available: {e}")
    print(f"   Please ensure CTGAN dependencies are installed")
except Exception as e:
    print(f"❌ Error during CTGAN demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 CTGAN Demo - Default Parameters
Training CTGAN with demo parameters...


Gen. (-0.35) | Discrim. (-0.32): 100%|██████████| 500/500 [00:08<00:00, 60.63it/s]


Generating 500 synthetic samples...
✅ CTGAN Demo completed successfully!
   - Training time: 14.20 seconds
   - Generated samples: 500
   - Original data shape: (500, 6)
   - Synthetic data shape: (500, 6)


#### 3.1.2 CTAB-GAN Demo

In [16]:
# Code Chunk ID: CHUNK_3_1_2_A
try:
    print("🔄 CTAB-GAN Demo - Default Parameters")
    print("=" * 50)
    
    # Check CTABGAN availability (imported from setup.py)
    if not CTABGAN_AVAILABLE:
        raise ImportError("CTAB-GAN not available - clone and install CTAB-GAN repository")
    
    # Initialize CTAB-GAN model (already defined in notebook)
    ctabgan_model = CTABGANModel()
    print("✅ CTAB-GAN model initialized successfully")
    
    # Record start time
    start_time = time.time()
    
    # Train the model with demo parameters
    print("🚀 Training CTAB-GAN model (epochs=500)...")
    ctabgan_model.fit(data, categorical_columns=get_categorical_columns_for_models(), target_column=target_column)
    
    # Record training time
    train_time = time.time() - start_time
    
    # Generate synthetic data
    print("🎯 Generating synthetic data...")
    synthetic_data_ctabgan = ctabgan_model.generate(len(data))
    
    # Display results
    print("✅ CTAB-GAN Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ctabgan)}")
    print(f"   - Original shape: {data.shape}")
    print(f"   - Synthetic shape: {synthetic_data_ctabgan.shape}")
    
    # Show sample of synthetic data with proper handling for both DataFrame and array
    print(f"\n📊 Sample of generated data:")
    if hasattr(synthetic_data_ctabgan, 'head'):
        # It's a DataFrame
        print(synthetic_data_ctabgan.head())
    else:
        # It's likely a numpy array
        print("First 5 rows of synthetic data:")
        print(synthetic_data_ctabgan[:5])
    print("=" * 50)
    
except ImportError as e:
    print(f"❌ CTAB-GAN not available: {e}")
    print(f"   Please ensure CTAB-GAN dependencies are installed")
    print(f"   Note: CTABGAN_AVAILABLE = {globals().get('CTABGAN_AVAILABLE', 'undefined')}")
except Exception as e:
    print(f"❌ Error during CTAB-GAN demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 CTAB-GAN Demo - Default Parameters
✅ CTAB-GAN model initialized successfully
🚀 Training CTAB-GAN model (epochs=500)...
[CTABGAN] Applying comprehensive data preprocessing...
[DATA_CLEANING] Processing 500 rows, 6 columns
[DATA_CLEANING] Categorical columns: []
[DATA_CLEANING] Data cleaning completed successfully
[DATA_CLEANING] Final shape: (500, 6)
[DATA_CLEANING] Data types: {'mean_radius': dtype('float64'), 'mean_texture': dtype('float64'), 'mean_perimeter': dtype('float64'), 'mean_area': dtype('float64'), 'mean_smoothness': dtype('float64'), 'diagnosis': dtype('int64')}
[CTABGAN] Using categorical columns: []
[CTABGAN] Data shape after preprocessing: (500, 6)
[CTABGAN] Training CTAB-GAN for 100 epochs...


100%|██████████| 100/100 [00:05<00:00, 18.40it/s]

[OK] CTAB-GAN training completed successfully
🎯 Generating synthetic data...
[CTABGAN] Generated 500 raw synthetic samples
[OK] CTAB-GAN generation completed: (500, 6)
✅ CTAB-GAN Demo completed successfully!
   - Training time: 6.26 seconds
   - Generated samples: 500
   - Original shape: (500, 6)
   - Synthetic shape: (500, 6)

📊 Sample of generated data:
   mean_radius  mean_texture  mean_perimeter    mean_area  mean_smoothness  \
0    20.966191     14.785382      162.085827  1156.340523         0.115475   
1    19.907957     15.516399       83.047544  1535.832580         0.123321   
2    14.041953     16.841392       86.108979   524.726066         0.074147   
3    11.387453     19.123558       73.771659   464.259091         0.084326   
4    11.158024     26.116538       79.793255   385.357643         0.078239   

   diagnosis  
0  -0.028035  
1   0.012007  
2   0.997588  
3   0.977941  
4   1.007682  


#### 3.1.3 CTAB-GAN+ Demo

In [17]:
# Code Chunk ID: CHUNK_3_1_3_A
try:
    print("🔄 CTAB-GAN+ Demo - Default Parameters")
    print("=" * 50)
    
    # Check CTABGAN+ availability with fallback
    try:
        ctabganplus_available = CTABGANPLUS_AVAILABLE
    except NameError:
        print("⚠️  CTABGANPLUS_AVAILABLE variable not defined - checking direct import...")
        try:
            # Try to check if CTABGANPLUS (the imported class) exists
            from model.ctabgan import CTABGAN as CTABGANPLUS
            ctabganplus_available = True
            print("✅ CTAB-GAN+ import check successful")
        except ImportError:
            ctabganplus_available = False
            print("❌ CTAB-GAN+ import check failed")
    
    if not ctabganplus_available:
        raise ImportError("CTAB-GAN+ not available - clone and install CTAB-GAN+ repository")
    
    # Initialize CTAB-GAN+ model with epochs parameter in constructor
    ctabganplus_model = CTABGANPlusModel(epochs=500)
    print("✅ CTAB-GAN+ model initialized successfully")
    
    # Record start time
    start_time = time.time()
    
    # Train the model (epochs already set in constructor)
    print("🚀 Training CTAB-GAN+ model (epochs=500)...")
    ctabganplus_model.fit(data)
    print("epochs attribute:", getattr(ctabganplus_model, "epochs", None))
    print("inner model epochs:", getattr(getattr(ctabganplus_model, "model", None), "epochs", None))
    
    # Record training time
    train_time = time.time() - start_time
    
    # Generate synthetic data
    print("🎯 Generating synthetic data...")
    synthetic_data_ctabganplus = ctabganplus_model.generate(len(data))
    
    # Display results
    print("✅ CTAB-GAN+ Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ctabganplus)}")
    print(f"   - Original shape: {data.shape}")
    print(f"   - Synthetic shape: {synthetic_data_ctabganplus.shape}")
    
    # Show sample of synthetic data with proper handling for both DataFrame and array
    print(f"\n📊 Sample of generated data:")
    if hasattr(synthetic_data_ctabganplus, 'head'):
        # It's a DataFrame
        print(synthetic_data_ctabganplus.head())
    else:
        # It's likely a numpy array
        print("First 5 rows of synthetic data:")
        print(synthetic_data_ctabganplus[:5])
    print("=" * 50)
    
except ImportError as e:
    print(f"❌ CTAB-GAN+ not available: {e}")
    print(f"   Please ensure CTAB-GAN+ dependencies are installed")
except Exception as e:
    print(f"❌ Error during CTAB-GAN+ demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 CTAB-GAN+ Demo - Default Parameters
✅ CTAB-GAN+ model initialized successfully
🚀 Training CTAB-GAN+ model (epochs=500)...
[CTABGAN+] Applying comprehensive data preprocessing...
[DATA_CLEANING] Processing 500 rows, 6 columns
[DATA_CLEANING] Categorical columns: []
[DATA_CLEANING] Data cleaning completed successfully
[DATA_CLEANING] Final shape: (500, 6)
[DATA_CLEANING] Data types: {'mean_radius': dtype('float64'), 'mean_texture': dtype('float64'), 'mean_perimeter': dtype('float64'), 'mean_area': dtype('float64'), 'mean_smoothness': dtype('float64'), 'diagnosis': dtype('int64')}
[CTABGAN+] Using categorical columns: []
[CTABGAN+] Data shape after preprocessing: (500, 6)
[CTABGAN+] Using Classification with target: diagnosis (2 unique values)
[CTABGAN+] Validating data for robust training...
[CTABGAN+] Initializing CTAB-GAN+ with validated parameters...
[CTABGAN+] Training CTAB-GAN+ (Enhanced) for 500 epochs...


100%|██████████| 500/500 [00:30<00:00, 16.47it/s]

Finished training in 31.138970375061035  seconds.
[OK] CTAB-GAN+ training completed successfully
epochs attribute: 500
inner model epochs: None
🎯 Generating synthetic data...
[CTABGAN+] Generated 500 raw synthetic samples
[OK] CTAB-GAN+ generation completed: (500, 6)
✅ CTAB-GAN+ Demo completed successfully!
   - Training time: 31.15 seconds
   - Generated samples: 500
   - Original shape: (500, 6)
   - Synthetic shape: (500, 6)

📊 Sample of generated data:
   mean_radius  mean_texture  mean_perimeter    mean_area  mean_smoothness  \
0    10.952663     19.039848       77.056452   461.291631         0.104536   
1    11.176964     19.489442       69.965355   228.193959         0.104957   
2    26.018520     17.430471      199.822323  2028.902073         0.120065   
3     8.871077     18.426415       66.249759   238.712115         0.099312   
4     9.895364     18.382368       68.262817   360.718783         0.110760   

   diagnosis  
0          1  
1          1  
2          0  
3         

#### 3.1.4 GANerAid Demo

In [18]:
# Code Chunk ID: CHUNK_3_1_4_A
try:
    print("🔄 GANerAid Demo - Default Parameters")
    print("=" * 50)
    
    # Check GANerAid availability with fallback
    try:
        ganeraid_available = GANERAID_AVAILABLE
        GANerAidModel  # Test if the class is available
    except NameError:
        print("⚠️ GANerAidModel not available - checking import...")
        try:
            # Try to import GANerAidModel
            from src.models.implementations.ganeraid_model import GANerAidModel
            ganeraid_available = True
            print("✅ GANerAidModel import successful")
        except ImportError:
            ganeraid_available = False
            print("❌ GANerAidModel import failed")
    
    if not ganeraid_available:
        raise ImportError("GANerAid not available - please install GANerAid dependencies")
    
    # Initialize GANerAid model
    ganeraid_model = GANerAidModel(device="cuda")
    print("✅ GANerAid model initialized successfully")
    
    # Define demo_samples variable for synthetic data generation
    demo_samples = len(data)  # Same size as original dataset
    
    # Train with minimal parameters for demo
    demo_params = {'epochs': 500, 'batch_size': 100}
    start_time = time.time()
    ganeraid_model.train(data, **demo_params)  # GANerAid uses train method
    train_time = time.time() - start_time
    
    # Generate synthetic data
    synthetic_data_ganeraid = ganeraid_model.generate(demo_samples)
    
    print(f"✅ GANerAid Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ganeraid)}")
    print(f"   - Original shape: {data.shape}")
    print(f"   - Synthetic shape: {synthetic_data_ganeraid.shape}")
    
    # Show sample of synthetic data with proper handling for both DataFrame and array
    print(f"\n📊 Sample of generated data:")
    if hasattr(synthetic_data_ganeraid, 'head'):
        # It's a DataFrame
        print(synthetic_data_ganeraid.head())
    else:
        # It's likely a numpy array
        print("First 5 rows of synthetic data:")
        print(synthetic_data_ganeraid[:5])
    print("=" * 50)
    
except ImportError as e:
    print(f"❌ GANerAid not available: {e}")
    print(f"   Please ensure GANerAid dependencies are installed")
except Exception as e:
    print(f"❌ Error during GANerAid demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 GANerAid Demo - Default Parameters
✅ GANerAid model initialized successfully
Initialized gan with the following parameters: 
lr_d = 0.0005
lr_g = 0.0005
hidden_feature_space = 200
batch_size = 100
nr_of_rows = 25
binary_noise = 0.2
Start training of gan for 500 epochs


100%|██████████| 500/500 [00:13<00:00, 35.79it/s, loss=d error: 0.5508515983819962 --- g error 2.8687078952789307] 


Generating 500 samples
✅ GANerAid Demo completed successfully!
   - Training time: 14.00 seconds
   - Generated samples: 500
   - Original shape: (500, 6)
   - Synthetic shape: (500, 6)

📊 Sample of generated data:
   mean_radius  mean_texture  mean_perimeter   mean_area  mean_smoothness  \
0    14.403003     18.377106       84.034378  578.741699         0.094768   
1    14.327751     18.171318       90.979584  777.762085         0.101520   
2    14.620867     14.645752       73.711662  661.960876         0.094687   
3    15.310673     16.955414       80.709518  586.482056         0.099789   
4    12.390819     13.965576       76.029060  621.915771         0.095159   

   diagnosis  
0          0  
1          1  
2          0  
3          0  
4          1  


#### 3.1.5 CopulaGAN Demo

In [23]:
# Code Chunk ID: CHUNK_3_1_5_A

try:
    print("🔄 CopulaGAN Demo - Default Parameters")
    print("=" * 50)

    from src.models.model_factory import ModelFactory
    copulagan_model = ModelFactory.create("copulagan", random_state=42)

    demo_params = {
        "epochs": 500,
        "batch_size": 100,
        "generator_dim": (128, 128),
        "discriminator_dim": (128, 128),
        "default_distribution": "beta",
        "enforce_min_max_values": True,
    }

    print("Training CopulaGAN with demo parameters...")

    start_time = time.time()
    discrete_columns = data.select_dtypes(include=["object"]).columns.tolist()

    import threading

    done = False
    err = None

    def _train():
        nonlocal_vars = None  # no-op; keeps this cell self-contained
        global done, err
        try:
            copulagan_model.train(data, discrete_columns=discrete_columns, **demo_params)
        except Exception as e:
            err = e
        finally:
            done = True

    t = threading.Thread(target=_train, daemon=True)
    t.start()

    while not done:
        print(f"[CopulaGAN] training... elapsed {time.time() - start_time:.1f}s")
        time.sleep(5)

    if err:
        raise err

    train_time = time.time() - start_time

    demo_samples = len(data)
    print(f"Generating {demo_samples} synthetic samples...")
    synthetic_data_copulagan = copulagan_model.generate(demo_samples)

    print("✅ CopulaGAN Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_copulagan)}")
    print(f"   - Original data shape: {data.shape}")
    print(f"   - Synthetic data shape: {synthetic_data_copulagan.shape}")
    print(f"   - Distribution used: {demo_params['default_distribution']}")

    demo_results_copulagan = {
        "model": copulagan_model,
        "synthetic_data": synthetic_data_copulagan,
        "training_time": train_time,
        "parameters_used": demo_params,
    }

except ImportError as e:
    print(f"❌ CopulaGAN not available: {e}")
    print("   Please ensure CopulaGAN dependencies are installed")

except Exception as e:
    print(f"❌ Error during CopulaGAN demo: {str(e)}")
    import traceback
    traceback.print_exc()


🔄 CopulaGAN Demo - Default Parameters
Training CopulaGAN with demo parameters...
[CopulaGAN] training... elapsed 0.0s
[CopulaGAN] training... elapsed 5.0s
[CopulaGAN] training... elapsed 10.0s
[CopulaGAN] training... elapsed 15.0s
[CopulaGAN] training... elapsed 20.0s
[CopulaGAN] training... elapsed 25.0s
[CopulaGAN] training... elapsed 30.0s
[CopulaGAN] training... elapsed 35.0s
[CopulaGAN] training... elapsed 40.0s
Generating 500 synthetic samples...
✅ CopulaGAN Demo completed successfully!
   - Training time: 45.04 seconds
   - Generated samples: 500
   - Original data shape: (500, 6)
   - Synthetic data shape: (500, 6)
   - Distribution used: beta


#### 3.1.6 TVAE Demo

In [20]:
# Code Chunk ID: CHUNK_3_1_6_A
try:
    print("🔄 TVAE Demo - Default Parameters")
    print("=" * 50)
    
    # Import and initialize TVAE model using ModelFactory
    from src.models.model_factory import ModelFactory
    
    tvae_model = ModelFactory.create("tvae", random_state=42)
    
    # Define demo parameters optimized for TVAE
    demo_params = {
        'epochs': 50,
        'batch_size': 100,
        'compress_dims': (128, 128),
        'decompress_dims': (128, 128),
        'l2scale': 1e-5,
        'loss_factor': 2,
        'learning_rate': 1e-3  # VAE-specific learning rate
    }
    
    # Train with demo parameters
    print("Training TVAE with demo parameters...")
    start_time = time.time()
    
    # Auto-detect discrete columns for TVAE
    discrete_columns = data.select_dtypes(include=['object']).columns.tolist()
    
    tvae_model.train(data, discrete_columns=discrete_columns, **demo_params)
    train_time = time.time() - start_time
    
    # Generate synthetic data
    demo_samples = len(data)  # Same size as original dataset
    print(f"Generating {demo_samples} synthetic samples...")
    synthetic_data_tvae = tvae_model.generate(demo_samples)
    
    print(f"✅ TVAE Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_tvae)}")
    print(f"   - Original data shape: {data.shape}")
    print(f"   - Synthetic data shape: {synthetic_data_tvae.shape}")
    print(f"   - VAE architecture: compress{demo_params['compress_dims']} → decompress{demo_params['decompress_dims']}")
    
    # Store for later use in comprehensive evaluation
    demo_results_tvae = {
        'model': tvae_model,
        'synthetic_data': synthetic_data_tvae,
        'training_time': train_time,
        'parameters_used': demo_params
    }
    
except ImportError as e:
    print(f"❌ TVAE not available: {e}")
    print(f"   Please ensure TVAE dependencies are installed")
except Exception as e:
    print(f"❌ Error during TVAE demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 TVAE Demo - Default Parameters
Training TVAE with demo parameters...
Generating 500 synthetic samples...
✅ TVAE Demo completed successfully!
   - Training time: 1.42 seconds
   - Generated samples: 500
   - Original data shape: (500, 6)
   - Synthetic data shape: (500, 6)
   - VAE architecture: compress(128, 128) → decompress(128, 128)


### 3.2 Batch Process

This section outputs figures and graphics from models run in 3.1. The next chunk will output results for whatever subsections of 3.1 were run. I.e., if there's need to debug one method, there's no need to run all subsections of 3.1.

In [24]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3 - BATCH EVALUATION FOR ALL TRAINED MODELS
# Standardized evaluation using enhanced batch evaluation system
# ============================================================================

print("🔍 SECTION 3 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

section3_results = evaluate_all_available_models(
    section_number=3,
    scope=globals(),  # Pass notebook scope to access synthetic data variables
    models_to_evaluate=None,  # Evaluate all available models
    real_data=None,  # Will use 'data' from scope
    target_col=None   # Will use 'target_column' from scope
)

if section3_results:
    print(f"\n🎉 SECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"📊 Evaluated {len(section3_results)} models successfully")
    print(f"📁 All results saved to organized folder structure")
    
    # Show quick summary of best performing models
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\n🏆 RANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\n⚠️ No models available for evaluation")
    print("   Train some models first in previous sections")

🔍 SECTION 3 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: breast-cancer-data
[INFO] Target column: diagnosis
[INFO] Variable pattern: standard
[INFO] Found 6 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] GANerAid
   [OK] CopulaGAN
   [OK] TVAE

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/breast-cancer-data/2026-01-07/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.759

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.046
   [CHART] Explained Variance (PC1, PC2): 0.641, 0.172

[3] DISTRIBUTION SIMILARITY
------

## 4: Hyperparameter Tuning for Each Model

### 4.1 Hyperparameter Optimization

#### 4.1.1 CTGAN Hyperparameter Optimization

In [25]:
# Code Chunk ID: CHUNK_4_1_1_A
# CTGAN Hyperparameter Optimization (Optuna)
#
# Assumptions:
# - `data` is a pandas DataFrame already loaded
# - `TARGET_COLUMN` is defined (or we fall back to last column)
# - `get_categorical_columns_for_models()` exists (optional; falls back to object dtype columns)
# - `enhanced_objective_function_v2(real_df, synth_df, target_col)` exists and returns (score, similarity, accuracy)

import optuna
import numpy as np
import pandas as pd
import time
import traceback

# Toggle this when you're ready:
RUN_MODE = "debug"   # "debug" (5 trials) or "full" (50 trials)
N_TRIALS = 5 if RUN_MODE == "debug" else 50


def ctgan_search_space(trial: optuna.Trial) -> dict:
    # Debug space: tighter to validate code quickly
    if RUN_MODE == "debug":
        epochs = trial.suggest_int("epochs", 200, 400, step=50)
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
        pac = trial.suggest_categorical("pac", [1, 2, 4, 8])
        gen_lr = trial.suggest_float("generator_lr", 1e-5, 1e-3, log=True)
        disc_lr = trial.suggest_float("discriminator_lr", 1e-5, 1e-3, log=True)
        disc_steps = trial.suggest_int("discriminator_steps", 1, 3)
        gen_decay = trial.suggest_float("generator_decay", 1e-8, 1e-4, log=True)
        disc_decay = trial.suggest_float("discriminator_decay", 1e-8, 1e-4, log=True)

        gen_dim = trial.suggest_categorical("generator_dim", [
            (128, 128),
            (256, 256),
            (256, 512, 256),
        ])
        disc_dim = trial.suggest_categorical("discriminator_dim", [
            (128, 128),
            (256, 256),
            (256, 512, 256),
        ])

    # Full space: wider for real optimization
    else:
        epochs = trial.suggest_int("epochs", 100, 1000, step=50)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])
        pac = trial.suggest_categorical("pac", [1, 2, 4, 8, 10])
        gen_lr = trial.suggest_float("generator_lr", 5e-6, 5e-3, log=True)
        disc_lr = trial.suggest_float("discriminator_lr", 5e-6, 5e-3, log=True)
        disc_steps = trial.suggest_int("discriminator_steps", 1, 5)
        gen_decay = trial.suggest_float("generator_decay", 1e-8, 1e-4, log=True)
        disc_decay = trial.suggest_float("discriminator_decay", 1e-8, 1e-4, log=True)

        gen_dim = trial.suggest_categorical("generator_dim", [
            (128, 128),
            (256, 256),
            (512, 256),
            (256, 512),
            (512, 512),
            (128, 256, 128),
            (256, 128, 64),
            (256, 512, 256),
        ])
        disc_dim = trial.suggest_categorical("discriminator_dim", [
            (128, 128),
            (256, 256),
            (256, 512),
            (512, 256),
            (128, 256, 128),
            (256, 512, 256),
        ])

    # Enforce CTGAN constraint: batch_size must be divisible by pac
    if batch_size % pac != 0:
        raise optuna.TrialPruned()

    return {
        "epochs": epochs,
        "batch_size": batch_size,
        "pac": pac,
        "generator_lr": gen_lr,
        "discriminator_lr": disc_lr,
        "generator_dim": gen_dim,
        "discriminator_dim": disc_dim,
        "discriminator_steps": disc_steps,
        "generator_decay": gen_decay,
        "discriminator_decay": disc_decay,
        # Don't tune logging flags
        "log_frequency": True,
        "verbose": False,
    }


def _import_ctgan_class():
    """
    Prefer ctgan package. Fallback to SDV CTGAN if needed.
    Returns (CTGANClass, backend_name).
    """
    try:
        from ctgan import CTGAN
        return CTGAN, "ctgan"
    except Exception:
        pass

    try:
        from sdv.single_table import CTGANSynthesizer
        return CTGANSynthesizer, "sdv.single_table.CTGANSynthesizer"
    except Exception:
        pass

    try:
        from sdv.tabular import CTGAN
        return CTGAN, "sdv.tabular.CTGAN"
    except Exception:
        pass

    raise ImportError("Could not import CTGAN from ctgan or sdv.")


def _fit_and_sample(model, df: pd.DataFrame, discrete_cols: list, n: int) -> pd.DataFrame:
    # Fit
    try:
        model.fit(df, discrete_columns=discrete_cols)
    except TypeError:
        model.fit(df)

    # Sample
    if hasattr(model, "sample"):
        try:
            return model.sample(n)
        except TypeError:
            return model.sample(num_rows=n)

    if hasattr(model, "generate"):
        return model.generate(n)

    raise AttributeError("Model has neither sample(...) nor generate(...).")


def ctgan_objective(trial: optuna.Trial) -> float:
    try:
        if "data" not in globals() or data is None:
            raise ValueError("Global `data` DataFrame is not defined.")

        target_col = globals().get("TARGET_COLUMN", None) or data.columns[-1]

        params = ctgan_search_space(trial)

        # discrete columns
        try:
            discrete_cols = get_categorical_columns_for_models()
        except Exception:
            discrete_cols = data.select_dtypes(include=["object"]).columns.tolist()

        CTGANClass, _backend = _import_ctgan_class()

        # Build model (fallback to minimal if kwargs not supported)
        try:
            model = CTGANClass(
                epochs=params["epochs"],
                batch_size=params["batch_size"],
                generator_lr=params["generator_lr"],
                discriminator_lr=params["discriminator_lr"],
                generator_dim=params["generator_dim"],
                discriminator_dim=params["discriminator_dim"],
                pac=params["pac"],
                discriminator_steps=params["discriminator_steps"],
                generator_decay=params["generator_decay"],
                discriminator_decay=params["discriminator_decay"],
                log_frequency=params["log_frequency"],
                verbose=params["verbose"],
            )
        except TypeError:
            model = CTGANClass(
                epochs=params["epochs"],
                batch_size=params["batch_size"],
                pac=params["pac"],
            )

        start = time.time()
        synthetic = _fit_and_sample(model, data, discrete_cols, n=len(data))
        score, similarity_score, accuracy_score = enhanced_objective_function_v2(data, synthetic, target_col)
        _elapsed = time.time() - start

        # Minimal per-trial output (one line)
        print(f"[CTGAN][trial {trial.number}] score={score:.4f} (sim={similarity_score:.4f}, acc={accuracy_score:.4f})")

        return float(score)

    except optuna.TrialPruned:
        raise
    except Exception as e:
        print(f"[CTGAN][trial {trial.number}] FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        return 0.0


ctgan_study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)

ctgan_study.optimize(ctgan_objective, n_trials=N_TRIALS)

best_trial = ctgan_study.best_trial
print(f"🏆 CTGAN best score: {best_trial.value:.4f} (trial {best_trial.number})")

ctgan_optimization_results = {
    "study": ctgan_study,
    "best_trial": best_trial,
}


[I 2026-01-07 17:18:57,809] A new study created in memory with name: no-name-860ea218-ade4-4292-8c8b-0496bac4a719
[I 2026-01-07 17:18:57,812] Trial 0 pruned. 


🎯 Starting CTGAN Hyperparameter Optimization
📊 Dataset: 500 rows x 6 cols

[CTGAN][trial 1] backend=ctgan epochs=250 batch_size=128 pac=8 gen_lr=2.83e-04


[I 2026-01-07 17:19:12,909] Trial 1 finished with value: 0.3455686968649755 and parameters: {'batch_size': 128, 'pac': 8, 'epochs': 250, 'generator_lr': 0.0002826640214114758, 'discriminator_lr': 7.178354965950512e-06, 'generator_dim': (512, 256), 'discriminator_dim': (512, 256), 'discriminator_steps': 1, 'generator_decay': 1.660673280591978e-08, 'discriminator_decay': 2.3431249991004486e-06}. Best is trial 1 with value: 0.3455686968649755.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3193
[OK] TRTS Evaluation: 2 scenarios, Average: 0.3850
[CHART] Combined Score: 0.3456 (Similarity: 0.3193, Accuracy: 0.3850)
[CTGAN][trial 1] score=0.3456 (similarity=0.3193, accuracy=0.3850) elapsed=15.1s

[CTGAN][trial 2] backend=ctgan epochs=750 batch_size=128 pac=2 gen_lr=1.53e-05


[I 2026-01-07 17:20:51,830] Trial 2 finished with value: 0.6784540163920206 and parameters: {'batch_size': 128, 'pac': 2, 'epochs': 750, 'generator_lr': 1.527644486528002e-05, 'discriminator_lr': 0.001478573329754852, 'generator_dim': (128, 256, 128), 'discriminator_dim': (256, 256), 'discriminator_steps': 4, 'generator_decay': 2.6457511266052228e-05, 'discriminator_decay': 6.695892771738391e-08}. Best is trial 2 with value: 0.6784540163920206.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5948
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8040
[CHART] Combined Score: 0.6785 (Similarity: 0.5948, Accuracy: 0.8040)
[CTGAN][trial 2] score=0.6785 (similarity=0.5948, accuracy=0.8040) elapsed=98.9s

[CTGAN][trial 3] backend=ctgan epochs=1000 batch_size=512 pac=2 gen_lr=7.04e-06
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4379


[I 2026-01-07 17:21:39,109] Trial 3 finished with value: 0.49954022542027265 and parameters: {'batch_size': 512, 'pac': 2, 'epochs': 1000, 'generator_lr': 7.0403928404827095e-06, 'discriminator_lr': 0.0006498977035959187, 'generator_dim': (128, 128), 'discriminator_dim': (128, 256, 128), 'discriminator_steps': 4, 'generator_decay': 8.026079969684924e-06, 'discriminator_decay': 5.2026292766659575e-06}. Best is trial 2 with value: 0.6784540163920206.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.5920
[CHART] Combined Score: 0.4995 (Similarity: 0.4379, Accuracy: 0.5920)
[CTGAN][trial 3] score=0.4995 (similarity=0.4379, accuracy=0.5920) elapsed=47.3s

[CTGAN][trial 4] backend=ctgan epochs=600 batch_size=128 pac=2 gen_lr=2.99e-05


[I 2026-01-07 17:23:17,784] Trial 4 finished with value: 0.5400759018155532 and parameters: {'batch_size': 128, 'pac': 2, 'epochs': 600, 'generator_lr': 2.9855736770198983e-05, 'discriminator_lr': 1.781378763343885e-05, 'generator_dim': (128, 128), 'discriminator_dim': (128, 256, 128), 'discriminator_steps': 5, 'generator_decay': 7.334941348687118e-06, 'discriminator_decay': 1.1191112101582088e-05}. Best is trial 2 with value: 0.6784540163920206.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5048
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5930
[CHART] Combined Score: 0.5401 (Similarity: 0.5048, Accuracy: 0.5930)
[CTGAN][trial 4] score=0.5401 (similarity=0.5048, accuracy=0.5930) elapsed=98.7s

📊 CTGAN optimization completed!
🏆 Best score: 0.6785 (trial 2)
Best parameters:
  - batch_size: 128
  - pac: 2
  - epochs: 750
  - generator_lr: 1.527644486528002e-05
  - discriminator_lr: 0.001478573329754852
  - generator_dim: (128, 256, 128)
  - discriminator_dim: (256, 256)
  - discriminator_steps: 4
  - generator_decay: 2.6457511266052228e-05
  - discriminator_decay: 6.695892771738391e-08

📈 Summary
  - completed: 4
  - failed:    0
  - pruned:    1

✅ CTGAN optimization data ready for analysis


#### 4.1.2 CTAB-GAN Hyperparameter Optimization

In [28]:
# Code Chunk ID: CHUNK_4_1_2_A
# CTAB-GAN Hyperparameter Optimization (Optuna) — UPDATED
#
# Addresses prior critiques:
# - Adds RUN_MODE toggle for 5-trial debug vs 50-trial full run
# - Fixes test_ratio (not a synthesis-quality hyperparameter) to a constant
# - Removes verbose prints
# - Removes pruner (MedianPruner) because CTAB-GAN training doesn't report intermediate steps here,
#   so pruning would not actually happen without instrumenting CTAB-GAN per-epoch metrics
# - Uses a single, consistent evaluator (TRTSEvaluator) and returns mean TRTS score

import optuna
import numpy as np
import pandas as pd

from src.models.model_factory import ModelFactory
from src.evaluation.trts_framework import TRTSEvaluator

RUN_MODE = "debug"   # "debug" (5 trials) or "full" (50 trials)
N_TRIALS = 5 if RUN_MODE == "debug" else 50


def ctabgan_search_space(trial):
    # Debug space: tighter to validate code quickly
    if RUN_MODE == "debug":
        epochs = trial.suggest_int("epochs", 200, 400, step=50)
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Full space: wider for real optimization
    else:
        epochs = trial.suggest_int("epochs", 200, 800, step=50)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])

    return {
        "epochs": epochs,
        "batch_size": batch_size,
        "test_ratio": 0.2,  # fixed
    }


def _coerce_target_dtype(real_df: pd.DataFrame, synth_df: pd.DataFrame, target_column: str) -> pd.DataFrame:
    """Ensure synthetic target dtype matches real target dtype (common issue for evaluators)."""
    synth_conv = synth_df.copy()

    if target_column in synth_conv.columns and target_column in real_df.columns:
        real_dtype = real_df[target_column].dtype
        synth_dtype = synth_conv[target_column].dtype

        if synth_dtype == "object" and real_dtype != "object":
            synth_conv[target_column] = pd.to_numeric(synth_conv[target_column], errors="coerce")
            if synth_conv[target_column].isna().any():
                mode_value = real_df[target_column].mode()[0]
                synth_conv[target_column] = synth_conv[target_column].fillna(mode_value)
            synth_conv[target_column] = synth_conv[target_column].astype(real_dtype)

    return synth_conv


def ctabgan_objective(trial):
    try:
        if "data" not in globals() or data is None:
            raise ValueError("Global `data` DataFrame is not defined.")
        if "target_column" not in globals() or target_column is None:
            raise ValueError("Global `target_column` is not defined.")

        params = ctabgan_search_space(trial)

        # Create + train
        model = ModelFactory.create("ctabgan", random_state=42)
        model.train(
            data,
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            test_ratio=params["test_ratio"],
        )

        # Generate
        synth = model.generate(len(data))
        synth_conv = _coerce_target_dtype(data, synth, target_column)

        # Evaluate (single evaluator)
        trts = TRTSEvaluator(random_state=42)
        trts_results = trts.evaluate_trts_scenarios(data, synth_conv, target_column=target_column)

        # Extract scores (0..1)
        if isinstance(trts_results, dict) and isinstance(trts_results.get("trts_scores", None), dict):
            scores = list(trts_results["trts_scores"].values())
        else:
            scores = [
                v for v in (trts_results.values() if isinstance(trts_results, dict) else [])
                if isinstance(v, (int, float)) and 0.0 <= v <= 1.0
            ]

        if not scores:
            return 0.0

        score = float(np.mean(scores))
        return max(0.0, min(1.0, score))

    except optuna.TrialPruned:
        raise
    except Exception:
        return 0.0


# NOTE: no pruner here (it won't prune without trial.report/should_prune inside training)
ctabgan_study = optuna.create_study(direction="maximize")
ctabgan_study.optimize(ctabgan_objective, n_trials=N_TRIALS)

ctabgan_best_params = ctabgan_study.best_params


[I 2026-01-07 17:32:57,323] A new study created in memory with name: no-name-b72fd1c9-d47d-430a-8266-6469ad300e48
100%|██████████| 300/300 [00:17<00:00, 17.01it/s]
[I 2026-01-07 17:33:15,804] Trial 0 finished with value: 0.8533333333333333 and parameters: {'epochs': 300, 'batch_size': 256}. Best is trial 0 with value: 0.8533333333333333.


Finished training in 18.395944595336914  seconds.


100%|██████████| 250/250 [00:14<00:00, 16.70it/s]
[I 2026-01-07 17:33:31,656] Trial 1 finished with value: 0.8466666666666667 and parameters: {'epochs': 250, 'batch_size': 256}. Best is trial 0 with value: 0.8533333333333333.


Finished training in 15.741442680358887  seconds.


100%|██████████| 250/250 [00:14<00:00, 17.05it/s]
[I 2026-01-07 17:33:47,287] Trial 2 finished with value: 0.845 and parameters: {'epochs': 250, 'batch_size': 128}. Best is trial 0 with value: 0.8533333333333333.


Finished training in 15.546326160430908  seconds.


100%|██████████| 400/400 [00:23<00:00, 16.97it/s]
[I 2026-01-07 17:34:11,707] Trial 3 finished with value: 0.8583333333333333 and parameters: {'epochs': 400, 'batch_size': 256}. Best is trial 3 with value: 0.8583333333333333.


Finished training in 24.33543109893799  seconds.


100%|██████████| 250/250 [00:14<00:00, 16.89it/s]
[I 2026-01-07 17:34:27,360] Trial 4 finished with value: 0.8066666666666666 and parameters: {'epochs': 250, 'batch_size': 512}. Best is trial 3 with value: 0.8583333333333333.


Finished training in 15.569714069366455  seconds.


#### 4.1.3 CTAB-GAN+ Hyperparameter Optimization

In [27]:
# Code Chunk ID: CHUNK_4_1_3_A
# CTAB-GAN+ Hyperparameter Optimization (Optuna)
# - Includes DEBUG (5 trials) vs FULL (50 trials) search spaces via a single flag
# - Keeps test_ratio fixed (not a synthesis-quality hyperparameter)
# - Uses one consistent evaluator (TRTSEvaluator)
# - Minimal prints

import optuna
import numpy as np
import pandas as pd

from src.models.model_factory import ModelFactory
from src.evaluation.trts_framework import TRTSEvaluator

# Toggle this when you're ready:
RUN_MODE = "debug"   # "debug" (5 trials) or "full" (50 trials)

N_TRIALS = 5 if RUN_MODE == "debug" else 50


def ctabganplus_search_space(trial):
    if RUN_MODE == "debug":
        epochs = trial.suggest_int("epochs", 200, 400, step=50)
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
        test_ratio = 0.2
    else:
        epochs = trial.suggest_int("epochs", 150, 1000, step=50)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])
        test_ratio = 0.2

    return {
        "epochs": epochs,
        "batch_size": batch_size,
        "test_ratio": test_ratio,
    }


def ctabganplus_objective(trial):
    try:
        params = ctabganplus_search_space(trial)

        model = ModelFactory.create("ctabganplus", random_state=42)
        model.train(
            data,
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            test_ratio=params["test_ratio"],
        )

        synthetic = model.generate(len(data))

        # Ensure target dtype matches real (if needed)
        synthetic_conv = synthetic.copy()
        if target_column in synthetic_conv.columns and target_column in data.columns:
            if synthetic_conv[target_column].dtype == "object" and data[target_column].dtype != "object":
                synthetic_conv[target_column] = pd.to_numeric(synthetic_conv[target_column], errors="coerce")
                if synthetic_conv[target_column].isna().any():
                    mode_value = data[target_column].mode()[0]
                    synthetic_conv[target_column] = synthetic_conv[target_column].fillna(mode_value)
                synthetic_conv[target_column] = synthetic_conv[target_column].astype(data[target_column].dtype)

        trts = TRTSEvaluator(random_state=42)
        trts_results = trts.evaluate_trts_scenarios(data, synthetic_conv, target_column=target_column)

        # Extract scores (0..1)
        if isinstance(trts_results, dict) and isinstance(trts_results.get("trts_scores", None), dict):
            scores = list(trts_results["trts_scores"].values())
        else:
            scores = [
                v for v in (trts_results.values() if isinstance(trts_results, dict) else [])
                if isinstance(v, (int, float)) and 0.0 <= v <= 1.0
            ]

        if not scores:
            return 0.0

        score = float(np.mean(scores))
        return max(0.0, min(1.0, score))

    except optuna.TrialPruned:
        raise
    except Exception:
        return 0.0


ctabganplus_study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)

ctabganplus_study.optimize(ctabganplus_objective, n_trials=N_TRIALS)

ctabganplus_best_params = ctabganplus_study.best_params

[I 2026-01-07 17:28:46,396] A new study created in memory with name: no-name-b530486f-8634-4b17-8b4d-32ce4d7a0392
100%|██████████| 350/350 [00:21<00:00, 16.65it/s]
[I 2026-01-07 17:29:08,250] Trial 0 finished with value: 0.8433333333333334 and parameters: {'epochs': 350, 'batch_size': 512}. Best is trial 0 with value: 0.8433333333333334.


Finished training in 21.779656887054443  seconds.


100%|██████████| 250/250 [00:24<00:00, 10.16it/s]
[I 2026-01-07 17:29:33,712] Trial 1 finished with value: 0.8883333333333333 and parameters: {'epochs': 250, 'batch_size': 128}. Best is trial 1 with value: 0.8883333333333333.


Finished training in 25.383636713027954  seconds.


100%|██████████| 200/200 [00:19<00:00, 10.15it/s]
[I 2026-01-07 17:29:54,258] Trial 2 finished with value: 0.8166666666666667 and parameters: {'epochs': 200, 'batch_size': 128}. Best is trial 1 with value: 0.8883333333333333.


Finished training in 20.466750144958496  seconds.


100%|██████████| 200/200 [00:19<00:00, 10.08it/s]
[I 2026-01-07 17:30:14,931] Trial 3 finished with value: 0.865 and parameters: {'epochs': 200, 'batch_size': 128}. Best is trial 1 with value: 0.8883333333333333.


Finished training in 20.59339165687561  seconds.


100%|██████████| 250/250 [00:25<00:00,  9.94it/s]
[I 2026-01-07 17:30:40,926] Trial 4 finished with value: 0.8766666666666667 and parameters: {'epochs': 250, 'batch_size': 128}. Best is trial 1 with value: 0.8883333333333333.


Finished training in 25.91696572303772  seconds.


#### 4.1.4 GANerAid Hyperparameter Optimization

In [38]:
# Code Chunk ID: CHUNK_4_1_4_A
# GANerAid Hyperparameter Optimization (Optuna) — UPDATED (sample only feasible triples)
#
# Goal: avoid 0.0 trials caused by infeasible (nr_of_rows, batch_size, hidden_feature_space) combos.
# Approach:
# - Precompute ALL feasible triples (nr_of_rows, batch_size, hidden_feature_space) given candidate lists + dataset size
# - Sample exactly one feasible triple per trial (guarantees divisibility + size constraints)
#
# Notes:
# - You can still get 0.0 if training/evaluation fails (exceptions/NaNs), but feasibility zeros should be eliminated.
# - GPU-safe: ModelFactory.create(..., device="cuda") when available
# - Minimal prints

import optuna
import numpy as np
import pandas as pd
import time

from src.models.model_factory import ModelFactory

RUN_MODE = "debug"   # "debug" (5 trials) or "full" (50 trials)
N_TRIALS = 5 if RUN_MODE == "debug" else 50


def _build_feasible_triples(dataset_size: int):
    if RUN_MODE == "debug":
        row_candidates = [4, 5, 8, 10, 16, 20, 25]
        batch_choices = [64, 100, 128, 200, 256]
        hidden_choices = [100, 150, 200, 300, 400]
    else:
        row_candidates = [4, 5, 8, 10, 16, 20, 25, 32, 40]
        batch_choices = [32, 64, 100, 128, 200, 256, 400, 500]
        hidden_choices = [100, 150, 200, 300, 400, 500, 600]

    triples = []
    for r in row_candidates:
        if r < 4 or r >= dataset_size:
            continue
        for b in batch_choices:
            if b % r != 0:
                continue
            for h in hidden_choices:
                if h % r != 0:
                    continue
                triples.append((r, b, h))
    return triples


def ganeraid_search_space(trial):
    if "data" not in globals() or data is None:
        return None
    dataset_size = len(data)

    feasible_triples = _build_feasible_triples(dataset_size)
    if not feasible_triples:
        return None

    # Sample a guaranteed-feasible triple
    nr_of_rows, batch_size, hidden_feature_space = trial.suggest_categorical(
        "feasible_triple", feasible_triples
    )

    if RUN_MODE == "debug":
        epochs = trial.suggest_int("epochs", 100, 300, step=50)
    else:
        epochs = trial.suggest_int("epochs", 100, 500, step=50)

    return {
        "epochs": epochs,
        "batch_size": batch_size,
        "nr_of_rows": nr_of_rows,
        "hidden_feature_space": hidden_feature_space,
        "lr_d": trial.suggest_float("lr_d", 1e-6, 5e-3, log=True),
        "lr_g": trial.suggest_float("lr_g", 1e-6, 5e-3, log=True),
        "binary_noise": trial.suggest_float("binary_noise", 0.05, 0.6),
        "generator_decay": trial.suggest_float("generator_decay", 1e-8, 1e-3, log=True),
        "discriminator_decay": trial.suggest_float("discriminator_decay", 1e-8, 1e-3, log=True),
        "dropout_generator": trial.suggest_float("dropout_generator", 0.0, 0.5),
        "dropout_discriminator": trial.suggest_float("dropout_discriminator", 0.0, 0.5),
    }


def ganeraid_objective(trial):
    try:
        if "data" not in globals() or data is None:
            return 0.0

        target_col = globals().get("TARGET_COLUMN", None) or data.columns[-1]
        if "enhanced_objective_function_v2" not in globals():
            return 0.0

        params = ganeraid_search_space(trial)
        if params is None:
            return 0.0

        print(
            f"[GANerAid][trial {trial.number}] "
            f"epochs={params['epochs']} bs={params['batch_size']} rows={params['nr_of_rows']} hidden={params['hidden_feature_space']}"
        )

        try:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            device = "cpu"

        model = ModelFactory.create("ganeraid", device=device, random_state=42)
        model.set_config(params)

        start = time.time()
        model.train(
            data,
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            nr_of_rows=params["nr_of_rows"],
            hidden_feature_space=params["hidden_feature_space"],
            lr_d=params["lr_d"],
            lr_g=params["lr_g"],
            binary_noise=params["binary_noise"],
            generator_decay=params["generator_decay"],
            discriminator_decay=params["discriminator_decay"],
        )
        elapsed = time.time() - start

        synth = model.generate(len(data))
        score, similarity_score, accuracy_score = enhanced_objective_function_v2(data, synth, target_col)

        if pd.isna(score) or (not np.isfinite(score)):
            return 0.0

        score = float(max(0.0, min(1.0, score)))
        print(f"[GANerAid][trial {trial.number}] score={score:.4f} elapsed={elapsed:.1f}s")
        return score

    except Exception:
        return 0.0


ganeraid_study = optuna.create_study(direction="maximize")
ganeraid_study.optimize(ganeraid_objective, n_trials=N_TRIALS)

_completed = [t for t in ganeraid_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
ganeraid_best_params = ganeraid_study.best_params if _completed else None


[I 2026-01-07 17:51:14,031] A new study created in memory with name: no-name-ea39e32b-1660-43bd-b19f-8f66c74a598f


[GANerAid][trial 0] epochs=150 bs=200 rows=20 hidden=100
Initialized gan with the following parameters: 
lr_d = 4.1475989309766356e-05
lr_g = 3.5012461022811037e-06
hidden_feature_space = 100
batch_size = 200
nr_of_rows = 20
binary_noise = 0.43951186910971757
Start training of gan for 150 epochs


100%|██████████| 150/150 [00:03<00:00, 40.53it/s, loss=d error: 0.9881204068660736 --- g error 0.7831284403800964]


Generating 500 samples
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.2875


[I 2026-01-07 17:51:18,043] Trial 0 finished with value: 0.32887674342915546 and parameters: {'feasible_triple': (20, 200, 100), 'epochs': 150, 'lr_d': 4.1475989309766356e-05, 'lr_g': 3.5012461022811037e-06, 'binary_noise': 0.43951186910971757, 'generator_decay': 7.438599264080617e-07, 'discriminator_decay': 0.0002729791747757378, 'dropout_generator': 0.1725073912491476, 'dropout_discriminator': 0.4744561871358236}. Best is trial 0 with value: 0.32887674342915546.


...

[OK] TRTS Evaluation: 2 scenarios, Average: 0.8080
[CHART] Combined Score: 0.5050 (Similarity: 0.3029, Accuracy: 0.8080)
[GANerAid][trial 3] score=0.5050 elapsed=3.7s
[GANerAid][trial 4] epochs=200 bs=200 rows=10 hidden=200
Initialized gan with the following parameters: 
lr_d = 4.175294484470155e-06
lr_g = 0.0005151868581627388
hidden_feature_space = 200
batch_size = 200
nr_of_rows = 10
binary_noise = 0.11513329093756824
Start training of gan for 200 epochs


100%|██████████| 200/200 [00:04<00:00, 49.75it/s, loss=d error: 1.4146389365196228 --- g error 0.7192818522453308]


Generating 500 samples
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3069


[I 2026-01-07 17:51:38,170] Trial 4 finished with value: 0.4245447689698947 and parameters: {'feasible_triple': (10, 200, 200), 'epochs': 200, 'lr_d': 4.175294484470155e-06, 'lr_g': 0.0005151868581627388, 'binary_noise': 0.11513329093756824, 'generator_decay': 4.232198849471849e-08, 'discriminator_decay': 0.000530261409827632, 'dropout_generator': 0.34454008544828135, 'dropout_discriminator': 0.4750292592989441}. Best is trial 3 with value: 0.5049523639840392.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.6010
[CHART] Combined Score: 0.4245 (Similarity: 0.3069, Accuracy: 0.6010)
[GANerAid][trial 4] score=0.4245 elapsed=4.0s


#### 4.1.5 CopulaGAN Hyperparameter Optimization

In [39]:
# Code Chunk ID: CHUNK_4_1_5_A
# CopulaGAN Hyperparameter Optimization (Optuna) — UPDATED
#
# Critique fixes applied:
# - RUN_MODE toggle ("debug"=5 trials, "full"=50 trials) with different search ranges
# - Replace deprecated suggest_loguniform -> suggest_float(log=True)
# - Reduce noisy prints: one line per trial + final summary
# - Generate N samples equal to len(data) (not a hard-coded 5000) to keep scoring comparable across models
# - Discrete columns detection: include object + category dtypes; if none, pass empty list
# - More realistic learning rate ranges for SDV CTGAN/CopulaGAN family
# - Keep batch_size candidates smaller in debug; broaden in full
# - Guard TARGET_COLUMN fallback (use last column if missing)

import optuna
import numpy as np
import pandas as pd
import time

from src.models.model_factory import ModelFactory

RUN_MODE = "debug"   # "debug" (5 trials) or "full" (50 trials)
N_TRIALS = 5 if RUN_MODE == "debug" else 50


def copulagan_search_space(trial):
    if RUN_MODE == "debug":
        epochs = trial.suggest_int("epochs", 100, 300, step=50)
        batch_size = trial.suggest_categorical("batch_size", [100, 200, 500])
        gen_lr = trial.suggest_float("generator_lr", 1e-5, 1e-3, log=True)
        disc_lr = trial.suggest_float("discriminator_lr", 1e-5, 1e-3, log=True)
        gen_decay = trial.suggest_float("generator_decay", 1e-8, 1e-4, log=True)
        disc_decay = trial.suggest_float("discriminator_decay", 1e-8, 1e-4, log=True)
    else:
        epochs = trial.suggest_int("epochs", 100, 800, step=50)
        batch_size = trial.suggest_categorical("batch_size", [100, 200, 500, 1000])
        gen_lr = trial.suggest_float("generator_lr", 5e-6, 5e-3, log=True)
        disc_lr = trial.suggest_float("discriminator_lr", 5e-6, 5e-3, log=True)
        gen_decay = trial.suggest_float("generator_decay", 1e-8, 1e-3, log=True)
        disc_decay = trial.suggest_float("discriminator_decay", 1e-8, 1e-3, log=True)

    return {
        "epochs": epochs,
        "batch_size": batch_size,
        "generator_lr": gen_lr,
        "discriminator_lr": disc_lr,
        "generator_decay": gen_decay,
        "discriminator_decay": disc_decay,
    }


def _detect_discrete_columns(df: pd.DataFrame):
    # CopulaGAN in SDV expects discrete columns (categoricals) to be identified
    discrete = []
    for c in df.columns:
        if str(df[c].dtype) == "object" or str(df[c].dtype) == "category":
            discrete.append(c)
    return discrete


def copulagan_objective(trial):
    try:
        if "data" not in globals() or data is None:
            return 0.0

        # Target fallback
        target_col = globals().get("TARGET_COLUMN", None)
        if not target_col:
            target_col = data.columns[-1]

        # Need scoring function available
        if "enhanced_objective_function_v2" not in globals():
            return 0.0

        params = copulagan_search_space(trial)

        discrete_columns = _detect_discrete_columns(data)

        model = ModelFactory.create("copulagan", random_state=42)

        print(f"[CopulaGAN][trial {trial.number}] epochs={params['epochs']} bs={params['batch_size']} disc_cols={len(discrete_columns)}")

        start = time.time()
        model.train(data, discrete_columns=discrete_columns, **params)
        elapsed = time.time() - start

        # Keep sample size consistent with real data
        synth = model.generate(len(data))

        score, sim, acc = enhanced_objective_function_v2(data, synth, target_col)

        if pd.isna(score) or (not np.isfinite(score)):
            return 0.0

        score = float(max(0.0, min(1.0, score)))
        print(f"[CopulaGAN][trial {trial.number}] score={score:.4f} elapsed={elapsed:.1f}s")
        return score

    except Exception:
        return 0.0


copulagan_study = optuna.create_study(direction="maximize")
copulagan_study.optimize(copulagan_objective, n_trials=N_TRIALS)

best_trial = copulagan_study.best_trial
best_params_copulagan = best_trial.params
best_score_copulagan = best_trial.value

copulagan_results = {
    "model_name": "CopulaGAN",
    "best_score": best_score_copulagan,
    "best_params": best_params_copulagan,
    "study": copulagan_study,
    "n_trials": len(copulagan_study.trials),
}


[I 2026-01-07 17:54:11,865] A new study created in memory with name: no-name-4d7149fd-f4f9-4edd-a172-951d72338a8c


[CopulaGAN][trial 0] epochs=150 bs=500 disc_cols=0


[I 2026-01-07 17:54:20,050] Trial 0 finished with value: 0.4173271372544256 and parameters: {'epochs': 150, 'batch_size': 500, 'generator_lr': 5.607653377465943e-05, 'discriminator_lr': 5.865940236910378e-05, 'generator_decay': 1.1828390540377156e-07, 'discriminator_decay': 6.072684256645742e-05}. Best is trial 0 with value: 0.4173271372544256.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3415
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5310
[CHART] Combined Score: 0.4173 (Similarity: 0.3415, Accuracy: 0.5310)
[CopulaGAN][trial 0] score=0.4173 elapsed=7.9s
[CopulaGAN][trial 1] epochs=250 bs=100 disc_cols=0


[I 2026-01-07 17:54:41,623] Trial 1 finished with value: 0.6186894117318416 and parameters: {'epochs': 250, 'batch_size': 100, 'generator_lr': 4.793509314498918e-05, 'discriminator_lr': 5.786420388371269e-05, 'generator_decay': 1.6947639069592583e-05, 'discriminator_decay': 1.9069560931822015e-07}. Best is trial 1 with value: 0.6186894117318416.


...

[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3729
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7930
[CHART] Combined Score: 0.5410 (Similarity: 0.3729, Accuracy: 0.7930)
[CopulaGAN][trial 2] score=0.5410 elapsed=13.1s
[CopulaGAN][trial 3] epochs=300 bs=500 disc_cols=0
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3608


[I 2026-01-07 17:55:02,613] Trial 3 finished with value: 0.47088298560350683 and parameters: {'epochs': 300, 'batch_size': 500, 'generator_lr': 1.0017347325601256e-05, 'discriminator_lr': 4.3130607571657856e-05, 'generator_decay': 4.3846663008299974e-08, 'discriminator_decay': 1.1783064579999543e-08}. Best is trial 1 with value: 0.6186894117318416.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.6360
[CHART] Combined Score: 0.4709 (Similarity: 0.3608, Accuracy: 0.6360)
[CopulaGAN][trial 3] score=0.4709 elapsed=7.3s
[CopulaGAN][trial 4] epochs=200 bs=200 disc_cols=0


[I 2026-01-07 17:55:10,836] Trial 4 finished with value: 0.5057548673125636 and parameters: {'epochs': 200, 'batch_size': 200, 'generator_lr': 1.1183052016335693e-05, 'discriminator_lr': 1.5146187814129671e-05, 'generator_decay': 7.132430717983347e-07, 'discriminator_decay': 2.5327432360987446e-06}. Best is trial 1 with value: 0.6186894117318416.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4229
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6300
[CHART] Combined Score: 0.5058 (Similarity: 0.4229, Accuracy: 0.6300)
[CopulaGAN][trial 4] score=0.5058 elapsed=8.0s


#### 4.1.6 TVAE Hyperparameter Optimization

In [40]:
# Code Chunk ID: CHUNK_4_1_6_A
# TVAE Hyperparameter Optimization (Optuna) — UPDATED
#
# Critique fixes applied:
# - Adds RUN_MODE toggle ("debug"=5 trials, "full"=50 trials) with different search ranges
# - Replace deprecated Optuna APIs:
#   - suggest_loguniform -> suggest_float(log=True)
#   - suggest_uniform    -> suggest_float
# - Remove verbose prints (1 line per trial + final summary)
# - Do NOT tune flags that are often unsupported/ignored by SDV TVAE wrappers:
#   - log_frequency, conditional_generation, verbose (removed)
# - Use model name "tvae" (ModelFactory expects lowercase in your repo)
# - Guard global target_column fallback
# - Removes MedianPruner (won't prune without trial.report/should_prune in-training)

import optuna
import numpy as np
import pandas as pd
import time

from src.models.model_factory import ModelFactory

RUN_MODE = "debug"   # "debug" (5 trials) or "full" (50 trials)
N_TRIALS = 5 if RUN_MODE == "debug" else 50


def tvae_search_space(trial):
    if RUN_MODE == "debug":
        epochs = trial.suggest_int("epochs", 100, 300, step=50)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
        learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
        embedding_dim = trial.suggest_int("embedding_dim", 32, 128, step=32)
        l2scale = trial.suggest_float("l2scale", 1e-6, 1e-3, log=True)
        dropout = trial.suggest_float("dropout", 0.0, 0.3)
        compress_dims = trial.suggest_categorical("compress_dims", [(128, 128), (256, 128), (256, 128, 64)])
        decompress_dims = trial.suggest_categorical("decompress_dims", [(128, 128), (64, 128), (64, 128, 256)])
    else:
        epochs = trial.suggest_int("epochs", 100, 800, step=50)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
        embedding_dim = trial.suggest_int("embedding_dim", 32, 256, step=32)
        l2scale = trial.suggest_float("l2scale", 1e-6, 1e-2, log=True)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        compress_dims = trial.suggest_categorical("compress_dims", [(128, 128), (256, 128), (256, 128, 64)])
        decompress_dims = trial.suggest_categorical("decompress_dims", [(128, 128), (64, 128), (64, 128, 256)])

    return {
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "compress_dims": compress_dims,
        "decompress_dims": decompress_dims,
        "embedding_dim": embedding_dim,
        "l2scale": l2scale,
        "dropout": dropout,
    }


def tvae_objective(trial):
    try:
        if "data" not in globals() or data is None:
            return 0.0
        target_col = globals().get("target_column", None) or globals().get("TARGET_COLUMN", None) or data.columns[-1]
        if "enhanced_objective_function_v2" not in globals():
            return 0.0

        params = tvae_search_space(trial)

        print(f"[TVAE][trial {trial.number}] epochs={params['epochs']} bs={params['batch_size']} lr={params['learning_rate']:.2e}")

        model = ModelFactory.create("tvae", random_state=42)
        model.set_config(params)

        start = time.time()
        model.train(data, **params)
        elapsed = time.time() - start

        synth = model.generate(len(data))
        score, sim, acc = enhanced_objective_function_v2(data, synth, target_col)

        if pd.isna(score) or (not np.isfinite(score)):
            return 0.0

        score = float(max(0.0, min(1.0, score)))
        print(f"[TVAE][trial {trial.number}] score={score:.4f} elapsed={elapsed:.1f}s")
        return score

    except Exception:
        return 0.0


tvae_study = optuna.create_study(direction="maximize")
tvae_study.optimize(tvae_objective, n_trials=N_TRIALS)

best_trial = tvae_study.best_trial
tvae_best_params = best_trial.params


[I 2026-01-07 17:56:33,443] A new study created in memory with name: no-name-f62f0fc0-6b67-4ca4-aa2d-488a3ae7d26e


[TVAE][trial 0] epochs=250 bs=256 lr=2.41e-03


[I 2026-01-07 17:56:38,928] Trial 0 finished with value: 0.6335268524334123 and parameters: {'epochs': 250, 'batch_size': 256, 'learning_rate': 0.0024094731571352107, 'embedding_dim': 96, 'l2scale': 9.076242222390655e-05, 'dropout': 0.25563522793080656, 'compress_dims': (256, 128, 64), 'decompress_dims': (128, 128)}. Best is trial 0 with value: 0.6335268524334123.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4725
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8750
[CHART] Combined Score: 0.6335 (Similarity: 0.4725, Accuracy: 0.8750)
[TVAE][trial 0] score=0.6335 elapsed=5.2s
[TVAE][trial 1] epochs=200 bs=256 lr=8.81e-04


[I 2026-01-07 17:56:43,416] Trial 1 finished with value: 0.6492832664795456 and parameters: {'epochs': 200, 'batch_size': 256, 'learning_rate': 0.0008807029064351657, 'embedding_dim': 128, 'l2scale': 2.8512025894295877e-05, 'dropout': 0.2677712138812267, 'compress_dims': (256, 128), 'decompress_dims': (128, 128)}. Best is trial 1 with value: 0.6492832664795456.


...

[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4491
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8960
[CHART] Combined Score: 0.6278 (Similarity: 0.4491, Accuracy: 0.8960)
[TVAE][trial 2] score=0.6278 elapsed=2.6s
[TVAE][trial 3] epochs=100 bs=64 lr=1.31e-03


[I 2026-01-07 17:56:52,443] Trial 3 finished with value: 0.6323780244791323 and parameters: {'epochs': 100, 'batch_size': 64, 'learning_rate': 0.0013109454090504266, 'embedding_dim': 32, 'l2scale': 1.0555459771986232e-06, 'dropout': 0.09415530258969769, 'compress_dims': (256, 128), 'decompress_dims': (64, 128, 256)}. Best is trial 1 with value: 0.6492832664795456.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4686
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8780
[CHART] Combined Score: 0.6324 (Similarity: 0.4686, Accuracy: 0.8780)
[TVAE][trial 3] score=0.6324 elapsed=5.9s
[TVAE][trial 4] epochs=150 bs=128 lr=2.53e-03


[I 2026-01-07 17:56:57,921] Trial 4 finished with value: 0.6351419463259276 and parameters: {'epochs': 150, 'batch_size': 128, 'learning_rate': 0.002530880089886268, 'embedding_dim': 128, 'l2scale': 0.00021506085364442592, 'dropout': 0.14878201208051609, 'compress_dims': (128, 128), 'decompress_dims': (64, 128, 256)}. Best is trial 1 with value: 0.6492832664795456.


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4839
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8620
[CHART] Combined Score: 0.6351 (Similarity: 0.4839, Accuracy: 0.8620)
[TVAE][trial 4] score=0.6351 elapsed=5.2s


In [41]:
# Create Optuna summary comparing all models
from src.visualization.section4 import create_all_models_optuna_summary

# Collect all studies (only include those that completed successfully)
studies_dict = {}
if 'ctgan_study' in locals():
    studies_dict['CTGAN'] = ctgan_study
if 'ctabgan_study' in locals():
    studies_dict['CTABGAN'] = ctabgan_study
if 'ctabganplus_study' in locals():
    studies_dict['CTABGAN+'] = ctabganplus_study
if 'ganeraid_study' in locals():
    studies_dict['GANerAid'] = ganeraid_study
if 'copulagan_study' in locals():
    studies_dict['CopulaGAN'] = copulagan_study
if 'tvae_study' in locals():
    studies_dict['TVAE'] = tvae_study

if studies_dict:
    create_all_models_optuna_summary(
        studies_dict=studies_dict,
        results_path=results_path,
        verbose=True
    )
    print(f"\n[OK] Optuna summary visualization created for {len(studies_dict)} models")
else:
    print("[WARNING] No Optuna studies available for summary visualization")


[VIZ] Saved: optuna_summary_all_models.png

[OK] Optuna summary visualization created for 6 models


### 4.2 Batch process 

This section outputs figures and graphics from models run in 4.1. The next chunk will output results for whatever subsections of 4.1 were run. I.e., if there's need to debug one method, there's no need to run all subsections of 4.1.

In [42]:
# Code Chunk ID: CHUNK_4_2_0_A
# ============================================================================
# SECTION 4 - BATCH HYPERPARAMETER OPTIMIZATION ANALYSIS
# ============================================================================

print("🔍 SECTION 4 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS")
print("=" * 80)
print()

# Use enhanced batch evaluation function from setup.py
# Following exact same pattern as CHUNK_018 (Section 3) - no module reload needed!
try:
    # Run batch analysis with file export for all models
    section4_batch_results = evaluate_hyperparameter_optimization_results(
        section_number=4,
        scope=globals(),  # Pass notebook scope to access study variables
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("✅ SECTION 4 HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS COMPLETED!")
    print("="*80)
    print(f"📊 Models processed: {len(section4_batch_results['summary_data'])}")
    print(f"📁 Results exported to: {section4_batch_results['results_dir']}")
    print(f"📋 Individual model analysis files:")
    print("   • Hyperparameter parameter_analysis.png plots")
    print("   • Optimization convergence_analysis.png graphs")
    print("   • Parameter correlation matrices")
    print("   • Best trial summary tables")
    print("   • Comprehensive optimization summary CSV")

    
except Exception as e:
    print(f"❌ Batch hyperparameter analysis failed: {str(e)}")
    print(f"🔍 Error details: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    print("\n⚠️  Falling back to individual chunk analysis if needed")

# ============================================================================
# SAVE BEST PARAMETERS TO CSV FOR SECTION 5 USE
# ============================================================================
print("\n" + "=" * 80)
print("💾 SAVING BEST PARAMETERS FROM SECTION 4 OPTIMIZATION")
print("=" * 80)

try:
    # Save all best parameters to CSV using setup.py function
    param_save_results = save_best_parameters_to_csv(
        scope=globals(),
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER
    )
    
    if param_save_results['success']:
        print(f"\n✅ Parameter saving completed successfully!")
        print(f"   • Files saved: {len(param_save_results['files_saved'])}")
        print(f"   • Parameter entries: {param_save_results['parameters_count']}")
        print(f"   • Models processed: {param_save_results['models_count']}")
        print(f"   • Directory: {param_save_results['results_dir']}")
        
        # Display saved files
        for file_path in param_save_results['files_saved']:
            print(f"     📁 {file_path.split('/')[-1]}")
    else:
        print(f"\n⚠️  Parameter saving completed with issues: {param_save_results['message']}")
        
except Exception as e:
    print(f"\n❌ Parameter saving failed: {str(e)}")
    print(f"   Section 5 will fall back to memory-based parameter retrieval")

print(f"\n📈 Section 4 hyperparameter optimization analysis complete!")
print("🏁 Ready for Section 5: Optimized model re-training")

🔍 SECTION 4 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS

[LOCATION] Using DATASET_IDENTIFIER from scope: breast-cancer-data
[TARGET] Final DATASET_IDENTIFIER for Section 4: breast-cancer-data

SECTION 4 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS
[FOLDER] Base results directory: results/breast-cancer-data/2026-01-07/Section-4
[TARGET] Target column: diagnosis
[CHART] Dataset identifier: breast-cancer-data


[SEARCH] 4.1.1: CTGAN Hyperparameter Optimization Analysis
------------------------------------------------------------
[OK] CTGAN optimization study found
[FOLDER] Model directory: results/breast-cancer-data/2026-01-07/Section-4/CTGAN
[SEARCH] ANALYZING CTGAN HYPERPARAMETER OPTIMIZATION
[CHART] 1. TRIAL DATA EXTRACTION AND PROCESSING
----------------------------------------
[OK] Extracted 5 trials for analysis
[CHART] 2. PARAMETER SPACE EXPLORATION ANALYSIS
----------------------------------------
   - Found 10 hyperparameters: ['params_batch_size', 'params_discriminator_decay', 

## Section 5: Final Model Comparison and Best-of-Best Selection

### 5.1 Re-run of models with best hyperparameters identified from Section 4

#### 5.1.1 Best CTGAN Model Evaluation

In [43]:
# Code Chunk ID: CHUNK_5_1_1_A
# Section 5.1: Best CTGAN Model Evaluation  
print("🏆 SECTION 5.1: BEST CTGAN MODEL EVALUATION")
print("=" * 60)

# ============================================================================
# LOAD BEST PARAMETERS FROM SECTION 4 (CSV + MEMORY FALLBACK)
# ============================================================================
print("📖 5.1.0 Loading best parameters from Section 4...")

try:
    # Load all best parameters using setup.py function
    param_data = load_best_parameters_from_csv(
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER,
        fallback_to_memory=True,
        scope=globals()
    )
    
    print(f"✅ Parameter loading completed from {param_data['source']}")
    print(f"   • Models available: {param_data['models_count']}")
    
    # Extract CTGAN parameters specifically
    loaded_ctgan_params = param_data['parameters'].get('ctgan', None)
    
except Exception as e:
    print(f"⚠️  Parameter loading failed: {str(e)}")
    print(f"   Falling back to direct memory access")
    loaded_ctgan_params = None

# 5.1.1 Retrieve Best Model Results from Section 4.1
print("\n📊 5.1.1 Retrieving best CTGAN results from Section 4.1...")

try:
    # Primary: Use loaded parameters if available
    if loaded_ctgan_params is not None:
        print(f"✅ Using loaded CTGAN parameters from {param_data['source']}")
        best_params = loaded_ctgan_params
        
        # Try to get additional metadata from memory if available
        if 'ctgan_study' in globals() and ctgan_study is not None and hasattr(ctgan_study, 'best_trial'):
            best_trial = ctgan_study.best_trial
            best_value = best_trial.value
            trial_number = best_trial.number
        else:
            # Use fallback values when memory unavailable  
            best_value = 0.0  # Will be recalculated during evaluation
            trial_number = "loaded_from_csv"
            print(f"   ⚠️  Memory study unavailable - using loaded parameters only")
        
    else:
        # Fallback: Direct memory access
        print(f"🔄 Falling back to direct memory access...")
        best_trial = ctgan_study.best_trial
        best_params = best_trial.params
        best_value = best_trial.value
        trial_number = best_trial.number
        print(f"✅ Using CTGAN parameters from memory")
    
    print(f"\n✅ Section 4.1 CTGAN optimization parameters retrieved!")
    print(f"   • Best Trial: #{trial_number}")
    print(f"   • Best Objective Score: {best_value:.4f}" if isinstance(best_value, (int, float)) else f"   • Best Objective Score: {best_value}")
    print(f"   • Parameter count: {len(best_params)}")
    
    # Display parameters
    print(f"\n📈 5.1.2 Best CTGAN configuration:")
    for param, value in best_params.items():
        if isinstance(value, float):
            print(f"   • {param}: {value:.4f}")
        else:
            print(f"   • {param}: {value}")
    
    print(f"🔍 Parameter source: {param_data.get('source', 'memory') if loaded_ctgan_params else 'memory'}")
    
    # ============================================================================
    # 5.1.3 TRAIN FINAL CTGAN MODEL WITH OPTIMIZED PARAMETERS
    # ============================================================================
    
    print(f"\n🔧 5.1.3 Training final CTGAN model with optimized parameters...")
    
    try:
        # Use ModelFactory pattern
        from src.models.model_factory import ModelFactory
        
        # Create CTGAN model
        final_ctgan_model = ModelFactory.create("ctgan", random_state=42)
        
        # Apply best parameters with defaults for missing values
        final_ctgan_params = {
            'epochs': best_params.get('epochs', 300),
            'batch_size': best_params.get('batch_size', 500),
            'generator_lr': best_params.get('generator_lr', 2e-4),
            'discriminator_lr': best_params.get('discriminator_lr', 2e-4),
            'generator_decay': best_params.get('generator_decay', 1e-6),
            'discriminator_decay': best_params.get('discriminator_decay', 1e-6),
            'pac': best_params.get('pac', 10),
            'verbose': best_params.get('verbose', True)
        }
        
        print("🔧 Training CTGAN with optimal hyperparameters...")
        for param, value in final_ctgan_params.items():
            print(f"   • Using {param}: {value}")
        
        # Train the model
        final_ctgan_model.train(data, **final_ctgan_params)
        print("✅ CTGAN training completed successfully!")
        
        # Generate synthetic data
        print("🎲 Generating synthetic data...")
        synthetic_ctgan_final = final_ctgan_model.generate(len(data))
        print(f"✅ Generated {len(synthetic_ctgan_final)} synthetic samples")
        
        # ============================================================================
        # 5.1.4 EVALUATE FINAL CTGAN MODEL PERFORMANCE
        # ============================================================================
        
        print("\n📊 5.1.4 Final CTGAN Model Evaluation...")
        
        # Use enhanced objective function for evaluation
        if 'enhanced_objective_function_v2' in globals():
            print("🎯 Enhanced objective function evaluation:")
            
            ctgan_final_score, ctgan_similarity, ctgan_accuracy = enhanced_objective_function_v2(
                real_data=data, 
                synthetic_data=synthetic_ctgan_final, 
                target_column=TARGET_COLUMN
            )
            
            print(f"\n✅ Final CTGAN Evaluation Results:")
            print(f"   • Overall Score: {ctgan_final_score:.4f}")
            print(f"   • Similarity Score: {ctgan_similarity:.4f} (60% weight)")  
            print(f"   • Accuracy Score: {ctgan_accuracy:.4f} (40% weight)")
            
            # Store results for Section 5.7 comparison
            ctgan_final_results = {
                'model_name': 'CTGAN',
                'objective_score': ctgan_final_score,
                'similarity_score': ctgan_similarity,
                'accuracy_score': ctgan_accuracy,
                'best_params': best_params,
                'parameter_source': param_data.get('source', 'memory') if loaded_ctgan_params else 'memory',
                'synthetic_data': synthetic_ctgan_final
            }
            
            print("🎯 CTGAN Final Assessment:")
            print(f"   • Production Ready: {'✅ Yes' if ctgan_final_score > 0.6 else '⚠️ Review Required'}")
            print(f"   • Recommended for: General-purpose tabular synthetic data generation")
            print(f"   • Final Score vs Optimization Score: {ctgan_final_score:.4f} vs {best_value:.4f}" if isinstance(best_value, (int, float)) else f"   • Final Score: {ctgan_final_score:.4f}")
            
        else:
            print("⚠️ Enhanced objective function not available - using basic evaluation")
            ctgan_final_results = {
                'model_name': 'CTGAN',
                'objective_score': best_value if isinstance(best_value, (int, float)) else 0.0,
                'best_params': best_params,
                'parameter_source': param_data.get('source', 'memory') if loaded_ctgan_params else 'memory',
                'synthetic_data': synthetic_ctgan_final
            }
                
    except Exception as train_error:
        print(f"❌ Failed to train final CTGAN model: {train_error}")
        import traceback
        traceback.print_exc()
        synthetic_ctgan_final = None
        ctgan_final_score = 0.0
        ctgan_final_results = {
            'model_name': 'CTGAN',
            'objective_score': 0.0,
            'error': str(train_error)
        }

except Exception as e:
    print(f"❌ Error accessing CTGAN parameters: {e}")
    print("   Please ensure Section 4.1 has been executed successfully or parameter CSV exists.")
    # Create empty results to prevent downstream errors
    synthetic_ctgan_final = None
    ctgan_final_results = {
        'model_name': 'CTGAN',
        'objective_score': 0.0,
        'error': str(e)
    }
    
print("\n" + "=" * 60)
print("✅ SECTION 5.1 COMPLETE: Best CTGAN model trained and evaluated")
print("🔄 Ready for Section 5.2: CTAB-GAN model training")

🏆 SECTION 5.1: BEST CTGAN MODEL EVALUATION
📖 5.1.0 Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/breast-cancer-data/2026-01-07/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded GANerAid: 9 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 6
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - ganeraid: 9 parameters
   - copulagan: 6 parameters
   - tvae: 8 parameters
✅ Parameter loading completed from CSV file
   • Models available: 6

📊 5.1.1 Retrieving best CTGAN results from Section 4.1...
✅ Using loaded CTGAN parameters from CSV file

✅ Section 4.1 CTGAN optimization parameters retrieved!
   • Best Trial: #2
   • Best Objective Score: 0.678

Gen. (-1.28) | Discrim. (0.08): 100%|██████████| 750/750 [00:16<00:00, 44.85it/s] 


✅ CTGAN training completed successfully!
🎲 Generating synthetic data...
✅ Generated 500 synthetic samples

📊 5.1.4 Final CTGAN Model Evaluation...
🎯 Enhanced objective function evaluation:
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4731
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8770
[CHART] Combined Score: 0.6346 (Similarity: 0.4731, Accuracy: 0.8770)

✅ Final CTGAN Evaluation Results:
   • Overall Score: 0.6346
   • Similarity Score: 0.4731 (60% weight)
   • Accuracy Score: 0.8770 (40% weight)
🎯 CTGAN Final Assessment:
   • Production Ready: ✅ Yes
   • Recommended for: General-purpose tabular synthetic data generation
   • Final Score vs Optimization Score: 0.6346 vs 0.6785

✅ SECTION 5.1 COMPLETE: Best CTGAN model trained and evaluated
🔄 Ready for Section 5.2: CTAB-GAN model training


#### 5.1.2 Best CTAB-GAN Model Evaluation

In [44]:
# Code Chunk ID: CHUNK_5_1_1_Aa

# Section 5.2: Best CTAB-GAN Model Evaluation
print("🏆 SECTION 5.2: BEST CTAB-GAN MODEL EVALUATION")
print("=" * 60)

# 5.2.1 Retrieve Best Model Results from Section 4.2
print("📊 5.2.1 Retrieving best CTAB-GAN results from Section 4.2...")

try:
    # Use unified parameter loading function
    ctabgan_params = get_model_parameters(
        model_name='ctab-gan',
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER,
        scope=globals()
    )
    
    if ctabgan_params is not None:
        best_params = ctabgan_params
        
        # Try to get additional metadata from memory if available
        if 'ctabgan_study' in globals() and ctabgan_study is not None:
            best_trial = ctabgan_study.best_trial
            best_objective_score = best_trial.value
            trial_number = best_trial.number
            print(f"✅ Section 4.2 CTAB-GAN optimization completed successfully!")
            print(f"   • Best Trial: #{trial_number}")
        else:
            # Use fallback values when memory unavailable
            best_objective_score = 0.0
            trial_number = "loaded_from_csv"
            print(f"✅ Section 4.2 CTAB-GAN parameters loaded from CSV!")
            print(f"   • Best Trial: #{trial_number}")
        
        print(f"   • Best Objective Score: {best_objective_score:.4f}" if isinstance(best_objective_score, (int, float)) else f"   • Best Objective Score: {best_objective_score}")
        print(f"   • Best Parameters:")
        for param, value in best_params.items():
            print(f"     - {param}: {value}")
        
        # 5.2.2 Train Final CTAB-GAN Model using Section 5.1 Pattern
        print("🔧 Training final CTAB-GAN model using Section 5.1 proven pattern with optimized parameters...")
        
        try:
            # Use the exact same ModelFactory pattern that works in Section 5.1
            from src.models.model_factory import ModelFactory
            
            # Create CTAB-GAN model using the working pattern
            final_ctabgan_model = ModelFactory.create("ctabgan", random_state=42)
            
            # Apply the best parameters found in Section 4.2 optimization
            final_ctabgan_params = {
                'epochs': best_params.get('epochs', 300),
                'batch_size': best_params.get('batch_size', 512),
                'lr': best_params.get('lr', 2e-4),
                'betas': best_params.get('betas', (0.5, 0.9)),
                'l2scale': best_params.get('l2scale', 1e-5),
                'mixed_precision': best_params.get('mixed_precision', False),
                'test_ratio': best_params.get('test_ratio', 0.20),
                'verbose': best_params.get('verbose', True)
            }
            
            print("🔧 Training CTAB-GAN with optimal hyperparameters...")
            for param, value in final_ctabgan_params.items():
                print(f"   • Using {param}: {value}")
            
            # Train the model with best parameters
            final_ctabgan_model.train(data, **final_ctabgan_params)
            print("✅ CTAB-GAN training completed successfully!")
            
            # Generate synthetic data
            print("📊 Generating synthetic data for evaluation...")
            synthetic_ctabgan_final = final_ctabgan_model.generate(len(data))
            print(f"✅ Generated {len(synthetic_ctabgan_final)} synthetic samples")
            
            # Evaluate using enhanced objective function
            if 'enhanced_objective_function_v2' in globals():
                print("🎯 CTAB-GAN Classification Performance Analysis:")
                
                ctabgan_final_score, ctabgan_similarity, ctabgan_accuracy = enhanced_objective_function_v2(
                    real_data=data, 
                    synthetic_data=synthetic_ctabgan_final, 
                    target_column=TARGET_COLUMN
                )
                
                print(f"✅ CTAB-GAN Final Results:")
                print(f"   • Overall Score: {ctabgan_final_score:.4f}")
                print(f"   • Similarity Score: {ctabgan_similarity:.4f}")  
                print(f"   • Accuracy Score: {ctabgan_accuracy:.4f}")
                
                # Store results for Section 5.7 comparison
                ctabgan_final_results = {
                    'model_name': 'CTAB-GAN',
                    'objective_score': ctabgan_final_score,
                    'similarity_score': ctabgan_similarity,
                    'accuracy_score': ctabgan_accuracy,
                    'best_params': best_params,
                    'synthetic_data': synthetic_ctabgan_final
                }
                
            else:
                print("⚠️ Enhanced objective function not available - using basic evaluation")
                ctabgan_final_results = {
                    'model_name': 'CTAB-GAN',
                    'objective_score': best_objective_score,
                    'best_params': best_params,
                    'synthetic_data': synthetic_ctabgan_final
                }
                
        except Exception as e:
            print(f"❌ CTAB-GAN training failed: {str(e)}")
            synthetic_ctabgan_final = None
            ctabgan_final_results = {
                'model_name': 'CTAB-GAN',
                'objective_score': 0.0,
                'error': str(e)
            }
        
    else:
        print("❌ CTAB-GAN study results not found - Section 4.2 may not have completed successfully")
        print("    Please ensure Section 4.2 has been executed before running Section 5.2")
        synthetic_ctabgan_final = None
        ctabgan_final_score = 0.0
        ctabgan_final_results = {
            'model_name': 'CTAB-GAN',
            'objective_score': 0.0,
            'error': 'Section 4.2 not completed'
        }
        
except Exception as e:
    print(f"❌ Error in Section 5.2 CTAB-GAN evaluation: {e}")
    import traceback
    traceback.print_exc()
    synthetic_ctabgan_final = None
    ctabgan_final_score = 0.0
    ctabgan_final_results = {
        'model_name': 'CTAB-GAN',
        'objective_score': 0.0,
        'error': str(e)
    }

print("✅ Section 5.2 CTAB-GAN evaluation completed!")
print("=" * 60)

🏆 SECTION 5.2: BEST CTAB-GAN MODEL EVALUATION
📊 5.2.1 Retrieving best CTAB-GAN results from Section 4.2...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/breast-cancer-data/2026-01-07/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded GANerAid: 9 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 6
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - ganeraid: 9 parameters
   - copulagan: 6 parameters
   - tvae: 8 parameters
[OK] CTAB-GAN parameters loaded from CSV file
✅ Section 4.2 CTAB-GAN optimization completed successfully!
   • Best Trial: #3
   • Best Objective Score: 0.8583
   • Best Parameters:
     - epochs: 400
     - batch_size: 256
🔧 Training final CTAB-GAN model using Sectio

100%|██████████| 400/400 [00:23<00:00, 16.72it/s]


Finished training in 24.695038557052612  seconds.
✅ CTAB-GAN training completed successfully!
📊 Generating synthetic data for evaluation...
✅ Generated 500 synthetic samples
🎯 CTAB-GAN Classification Performance Analysis:
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4565
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8940
[CHART] Combined Score: 0.6315 (Similarity: 0.4565, Accuracy: 0.8940)
✅ CTAB-GAN Final Results:
   • Overall Score: 0.6315
   • Similarity Score: 0.4565
   • Accuracy Score: 0.8940
✅ Section 5.2 CTAB-GAN evaluation completed!


#### 5.1.3 Best CTAB-GAN+ Model Evaluation

In [45]:
# Code Chunk ID: CHUNK_5_1_3_A
# ============================================================================
# Section 5.3: Best CTAB-GAN+ Model Evaluation - FIXED IMPLEMENTATION
# ============================================================================
# Using Section 4.3 optimized hyperparameters with proven ModelFactory pattern

print("🏆 SECTION 5.3: BEST CTAB-GAN+ MODEL EVALUATION")
print("=" * 80)

try:
    # Step 1: Retrieve Section 4.3 CTAB-GAN+ optimization results
    if 'ctabganplus_study' in globals():
        best_trial = ctabganplus_study.best_trial
        best_params = best_trial.params
        best_objective_score = best_trial.value
        
        print(f"✅ Retrieved Section 4.3 CTAB-GAN+ optimization results")
        print(f"   • Best Trial: #{best_trial.number}")
        print(f"   • Best Objective Score: {best_objective_score:.4f}")
        print(f"   • Parameters: {len(best_params)} hyperparameters")
        
        # Display best parameters
        print(f"\n📊 Best CTAB-GAN+ Hyperparameters:")
        print("-" * 40)
        for param, value in best_params.items():
            if isinstance(value, float):
                print(f"   • {param}: {value:.4f}")
            else:
                print(f"   • {param}: {value}")
                
    else:
        print("⚠️ CTAB-GAN+ optimization results not found - using fallback parameters")
        # Fallback CTAB-GAN+ parameters (basic working configuration)
        best_params = {
            'epochs': 100,
            'batch_size': 128,
            'lr_generator': 1e-4,
            'lr_discriminator': 2e-4,
            'beta_1': 0.5,
            'beta_2': 0.9,
            'lambda_gp': 10,
            'pac': 1
        }
        best_objective_score = None
        print(f"   Using fallback parameters: {best_params}")

    # Step 2: Create CTAB-GAN+ model using proven ModelFactory pattern (SAME AS SECTION 5.2)
    print(f"\n🏗️ Creating CTAB-GAN+ model using ModelFactory...")
    from src.models.model_factory import ModelFactory
    
    # CRITICAL FIX: Use the exact same ModelFactory pattern that works in Section 5.1 & 5.2
    final_ctabganplus_model = ModelFactory.create("ctabganplus", random_state=42)
    print(f"✅ CTAB-GAN+ model created successfully")
    
    # Step 3: Train using the correct method name: .train() (NOT .fit())
    print(f"\n🚀 Training CTAB-GAN+ model with optimized hyperparameters...")
    print(f"   • Data shape: {data.shape}")
    print(f"   • Target column: '{TARGET_COLUMN}'")
    print(f"   • Training with Section 4.3 parameters")
    
    # Store final parameters for results tracking
    final_ctabganplus_params = best_params.copy()
    
    # CRITICAL FIX: Train using .train() method (proven pattern from Sections 5.1 & 5.2)
    final_ctabganplus_model.train(data, **final_ctabganplus_params)
    print(f"✅ CTAB-GAN+ model training completed successfully!")
    
    # Step 4: Generate synthetic data using the correct method: .generate()
    print(f"\n📊 Generating synthetic data for evaluation...")
    synthetic_ctabganplus_final = final_ctabganplus_model.generate(len(data))
    print(f"✅ Synthetic data generated successfully!")
    print(f"   • Synthetic data shape: {synthetic_ctabganplus_final.shape}")
    print(f"   • Columns match: {list(synthetic_ctabganplus_final.columns) == list(data.columns)}")
    
    # Step 5: Quick evaluation using enhanced objective function (NO IMPORT - function in globals)
    if 'enhanced_objective_function_v2' in globals():
        ctabganplus_final_score, ctabganplus_similarity, ctabganplus_accuracy = enhanced_objective_function_v2(
            real_data=data, 
            synthetic_data=synthetic_ctabganplus_final, 
            target_column=TARGET_COLUMN
        )
        
        print(f"\n📊 CTAB-GAN+ Enhanced Objective Function v2 Results:")
        print(f"   • Final Combined Score: {ctabganplus_final_score:.4f}")
        print(f"   • Statistical Similarity (60%): {ctabganplus_similarity:.4f}")
        print(f"   • Classification Accuracy (40%): {ctabganplus_accuracy:.4f}")
    else:
        print("⚠️ Enhanced objective function not available - using basic metrics")
        ctabganplus_final_score = 0.5  # Fallback score
        ctabganplus_similarity = 0.5
        ctabganplus_accuracy = 0.5
    
    # Store results for Section 5.7 comparative analysis
    ctabganplus_final_results = {
        'model_name': 'CTAB-GAN+',
        'objective_score': ctabganplus_final_score,
        'similarity_score': ctabganplus_similarity,
        'accuracy_score': ctabganplus_accuracy,
        'final_combined_score': ctabganplus_final_score,
        'sections_completed': ['5.3.1'],
        'evaluation_method': 'section_5_1_pattern',
        'section_4_optimization': best_objective_score is not None,
        'best_section_4_score': best_objective_score
    }
    
    print(f"\n✅ SECTION 5.3 COMPLETED SUCCESSFULLY!")
    print(f"🎯 CTAB-GAN+ evaluation completed using Section 4.3 optimized parameters")
    print(f"📊 Results ready for Section 5.7 comparative analysis")
    print("-" * 80)

except Exception as e:
    print(f"❌ CTAB-GAN+ evaluation failed: {str(e)}")
    import traceback
    traceback.print_exc()
    # Set fallback for subsequent sections
    synthetic_ctabganplus_final = None
    ctabganplus_final_results = {'error': str(e), 'evaluation_failed': True}

🏆 SECTION 5.3: BEST CTAB-GAN+ MODEL EVALUATION
✅ Retrieved Section 4.3 CTAB-GAN+ optimization results
   • Best Trial: #1
   • Best Objective Score: 0.8883
   • Parameters: 2 hyperparameters

📊 Best CTAB-GAN+ Hyperparameters:
----------------------------------------
   • epochs: 250
   • batch_size: 128

🏗️ Creating CTAB-GAN+ model using ModelFactory...
✅ CTAB-GAN+ model created successfully

🚀 Training CTAB-GAN+ model with optimized hyperparameters...
   • Data shape: (500, 6)
   • Target column: 'diagnosis'
   • Training with Section 4.3 parameters


100%|██████████| 250/250 [00:24<00:00, 10.13it/s]


Finished training in 25.447794437408447  seconds.
✅ CTAB-GAN+ model training completed successfully!

📊 Generating synthetic data for evaluation...
✅ Synthetic data generated successfully!
   • Synthetic data shape: (500, 6)
   • Columns match: True
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4971
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8860
[CHART] Combined Score: 0.6527 (Similarity: 0.4971, Accuracy: 0.8860)

📊 CTAB-GAN+ Enhanced Objective Function v2 Results:
   • Final Combined Score: 0.6527
   • Statistical Similarity (60%): 0.4971
   • Classification Accuracy (40%): 0.8860

✅ SECTION 5.3 COMPLETED SUCCESSFULLY!
🎯 CTAB-GAN+ evaluation completed using Section 4.3 optimized parameters
📊 Results ready for Section 5.7 comparative analysis
--------------------------------------------------------------------------------


#### Section 5.1.4 BEST GANerAid MODEL

In [50]:
# Code Chunk ID: CHUNK_5_1_4_A
# ============================================================================
# Section 5.4.1: Best GANerAid Model Training — UPDATED (GPU-safe)
# ============================================================================
# Fix:
# - Instantiate GANerAid via ModelFactory with device="cuda" when available.
#   (Otherwise wrapper defaults to CPU and you get cuda input vs cpu params.)
# - Do NOT pass device into train(); wrapper sets it in __init__.

print("🏆 SECTION 5.4.1: BEST GANerAid MODEL TRAINING")
print("=" * 80)

try:
    # Step 1: Retrieve Section 4.4 GANerAid optimization results
    if "ganeraid_study" in globals() and ganeraid_study is not None:
        best_trial = ganeraid_study.best_trial
        final_ganeraid_params = dict(best_trial.params)
        best_objective_score = best_trial.value
        print("✅ Retrieved Section 4.4 GANerAid optimization results")
        print(f"   • Best Trial: #{best_trial.number}")
        print(f"   • Best Objective Score: {best_objective_score:.4f}")
        print(f"   • Parameters: {len(final_ganeraid_params)} hyperparameters")
    else:
        print("⚠️ GANerAid optimization results not found - using fallback parameters")
        final_ganeraid_params = {
            "epochs": 100,
            "batch_size": 128,
            "nr_of_rows": 8,
            "hidden_feature_space": 200,
            "lr_d": 5e-4,
            "lr_g": 5e-4,
            "binary_noise": 0.2,
        }
        best_objective_score = None

    # Ensure target column exists
    if "TARGET_COLUMN" not in globals() or TARGET_COLUMN is None:
        TARGET_COLUMN = data.columns[-1]

    # Select device
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        device = "cpu"

    # Step 2: Create GANerAid model using ModelFactory (GPU-safe)
    print("\n🏗️ Creating GANerAid model using ModelFactory...")
    from src.models.model_factory import ModelFactory
    final_ganeraid_model = ModelFactory.create("ganeraid", device=device, random_state=42)
    print(f"✅ GANerAid model created successfully (device={device})")

    # Step 3: Train using .train()
    print("\n🚀 Training GANerAid model with optimized hyperparameters...")
    final_ganeraid_model.train(data, **final_ganeraid_params)
    print("✅ GANerAid model training completed successfully!")

    # Step 4: Generate synthetic data
    synthetic_ganeraid_final = final_ganeraid_model.generate(len(data))
    print(f"✅ GANerAid synthetic data generated: {synthetic_ganeraid_final.shape}")

    # Step 5: Quick evaluation
    if "enhanced_objective_function_v2" in globals():
        ganeraid_final_score, ganeraid_similarity, ganeraid_accuracy = enhanced_objective_function_v2(
            real_data=data,
            synthetic_data=synthetic_ganeraid_final,
            target_column=TARGET_COLUMN,
        )
    else:
        ganeraid_final_score, ganeraid_similarity, ganeraid_accuracy = 0.0, 0.0, 0.0

    ganeraid_final_results = {
        "model_name": "GANerAid",
        "objective_score": ganeraid_final_score,
        "similarity_score": ganeraid_similarity,
        "accuracy_score": ganeraid_accuracy,
        "final_combined_score": ganeraid_final_score,
        "sections_completed": ["5.4.1"],
        "section_4_optimization": best_objective_score is not None,
        "best_section_4_score": best_objective_score,
        "optimized_params": final_ganeraid_params,
        "device": device,
    }

    print("\n✅ SECTION 5.4.1 - GANerAid MODEL TRAINING COMPLETED!")
    print("-" * 80)

except Exception as e:
    print(f"❌ GANerAid training failed: {str(e)}")
    import traceback
    traceback.print_exc()
    synthetic_ganeraid_final = None
    ganeraid_final_results = {"error": str(e), "training_failed": True}

🏆 SECTION 5.4.1: BEST GANerAid MODEL TRAINING
✅ Retrieved Section 4.4 GANerAid optimization results
   • Best Trial: #3
   • Best Objective Score: 0.5050
   • Parameters: 9 hyperparameters

🏗️ Creating GANerAid model using ModelFactory...
✅ GANerAid model created successfully (device=cuda)

🚀 Training GANerAid model with optimized hyperparameters...
Initialized gan with the following parameters: 
lr_d = 0.00021840492690058822
lr_g = 0.0009549916375363346
hidden_feature_space = 200
batch_size = 100
nr_of_rows = 25
binary_noise = 0.1347650032308832
Start training of gan for 100 epochs


100%|██████████| 100/100 [00:02<00:00, 35.27it/s, loss=d error: 1.5599858164787292 --- g error 0.4857553541660309]


✅ GANerAid model training completed successfully!
Generating 500 samples
✅ GANerAid synthetic data generated: (500, 6)
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3393
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8600
[CHART] Combined Score: 0.5476 (Similarity: 0.3393, Accuracy: 0.8600)

✅ SECTION 5.4.1 - GANerAid MODEL TRAINING COMPLETED!
--------------------------------------------------------------------------------


#### 5.1.5: Best CopulaGAN Model

In [47]:
# Code Chunk ID: CHUNK_5_1_5_A
# ============================================================================
# Section 5.5: Best CopulaGAN Model Evaluation
# ============================================================================
# Using Section 4.5 optimized hyperparameters with proven ModelFactory pattern

# Import ModelFactory for CopulaGAN creation
from src.models.model_factory import ModelFactory

print("🎯 ============================================================================")
print("🎯 Section 5.5: CopulaGAN Final Model Training & Evaluation")
print("🎯 ============================================================================")

try:
    # Load all best parameters using setup.py function
    param_data = load_best_parameters_from_csv(
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER,
        fallback_to_memory=True,
        scope=globals()
    )
    
    # Extract CopulaGAN parameters specifically
    loaded_copulagan_params = param_data['parameters'].get('copulagan', None)
    
    if loaded_copulagan_params:
        print(f"📋 Loaded CopulaGAN parameters from {param_data.get('source', 'unknown')}:")
        for param, value in loaded_copulagan_params.items():
            print(f"   • {param}: {value}")
        # Use all optimized parameters from Section 4.5
        best_params = loaded_copulagan_params.copy()
        param_source = param_data.get('source', 'CSV')
    else:
        print("⚠️ CopulaGAN optimization results not found - using fallback parameters")
        best_params = {'epochs': 500, 'batch_size': 100}  # Use Section 3 proven values
        param_source = 'fallback'
    
    print(f"\n🔧 Preprocessing data for CopulaGAN...")
    data_preprocessed = data.copy()
    print(f"   ✅ Data preprocessing completed: {data_preprocessed.shape}")
    print(f"   • Missing values: {data_preprocessed.isnull().sum().sum()}")
    
    # Show data types
    dtype_counts = data_preprocessed.dtypes.value_counts()
    print(f"   • Data types: {dict(dtype_counts)}")
    
    print(f"\n🏗️ Creating CopulaGAN model using ModelFactory...")
    final_copulagan_model = ModelFactory.create("copulagan", random_state=42)
    print("✅ CopulaGAN model created successfully")
    
    print(f"\n🚀 Training CopulaGAN model with optimized hyperparameters...")
    print(f"   • Using parameters: {best_params}")
    
    # Train the model without discrete_columns parameter - like working notebooks
    final_copulagan_model.train(data_preprocessed, **best_params)
    
    print("✅ CopulaGAN model training completed successfully")
    
    # Generate synthetic data
    print(f"\n🎲 Generating {len(data_preprocessed)} synthetic samples...")
    synthetic_copulagan_final = final_copulagan_model.generate(len(data_preprocessed))
    print(f"   ✅ Generated synthetic data: {synthetic_copulagan_final.shape}")
    
    # Comprehensive evaluation with enhanced objective function - using correct parameters
    print(f"\n📊 Comprehensive model evaluation...")
    copulagan_final_score, copulagan_similarity, copulagan_accuracy = enhanced_objective_function_v2(
        real_data=data_preprocessed,
        synthetic_data=synthetic_copulagan_final,
        target_column=TARGET_COLUMN
    )
    
    print(f"\n🎯 === CopulaGAN Final Results (Section 5.5) ===")
    print(f"📈 Combined Score: {copulagan_final_score:.4f}")
    print(f"📊 Similarity Score: {copulagan_similarity:.4f}")
    print(f"🎯 Accuracy Score: {copulagan_accuracy:.4f}")
    print(f"⚙️ Best Parameters: {best_params}")
    print(f"📁 Parameter Source: {param_source}")
    print(f"=====================================")
    
    # Store results for Section 5.2 batch processing
    copulagan_final_results = {
        'model_name': 'CopulaGAN',
        'combined_score': copulagan_final_score,
        'similarity_score': copulagan_similarity,
        'accuracy_score': copulagan_accuracy,
        'best_params': best_params,
        'parameter_source': param_data.get('source', 'memory'),
        'synthetic_data': synthetic_copulagan_final
    }
    
    print(f"💾 Results stored for Section 5.2 batch processing")
    
except Exception as e:
    error_msg = str(e)
    print(f"❌ CopulaGAN model creation/training failed: {error_msg}")
    print(f"   This may be due to CopulaGAN compatibility issues")
    
    # Fallback handling - store minimal results
    copulagan_final_results = {
        'model_name': 'CopulaGAN',
        'combined_score': 0.0,
        'similarity_score': 0.0,
        'accuracy_score': 0.0,
        'best_params': {'epochs': 500, 'batch_size': 100},
        'parameter_source': 'fallback',
        'error': error_msg,
        'synthetic_data': None
    }
    
    print(f"💾 Fallback results stored for Section 5.2 batch processing")

🎯 ============================================================================
🎯 Section 5.5: CopulaGAN Final Model Training & Evaluation
🎯 ============================================================================
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/breast-cancer-data/2026-01-07/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded GANerAid: 9 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 6
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - ganeraid: 9 parameters
   - copulagan: 6 parameters
   - tvae: 8 parameters
📋 Loaded CopulaGAN parameters from CSV file:
   • epochs: 250
   • batch_size: 100
   • generator_lr: 4.793509314498918e-05
   • discriminator_lr: 5.786420388

#### 5.1.6: Best TVAE Model Evaluation 

In [48]:
# Code Chunk ID: CHUNK_5_1_6_A
# ============================================================================
# Section 5.6.1: Best TVAE Model Training
# ============================================================================
# Using Section 4.6 optimized hyperparameters with proven ModelFactory pattern

print("🏆 SECTION 5.6.1: BEST TVAE MODEL TRAINING")
print("=" * 80)

try:
    # Step 1: Retrieve Section 4.6 TVAE optimization results
    if 'tvae_study' in globals():
        best_trial = tvae_study.best_trial
        final_tvae_params = best_trial.params
        best_objective_score = best_trial.value
        
        print(f"✅ Retrieved Section 4.6 TVAE optimization results")
        print(f"   • Best Trial: #{best_trial.number}")
        print(f"   • Best Objective Score: {best_objective_score:.4f}")
        print(f"   • Parameters: {len(final_tvae_params)} hyperparameters")
        
    else:
        print("⚠️ TVAE optimization results not found - using fallback parameters")
        # Fallback TVAE parameters
        final_tvae_params = {
            'epochs': 100,
            'batch_size': 128,
            'lr': 1e-4,
            'compress_dims': [128, 64],
            'decompress_dims': [64, 128]
        }
        best_objective_score = None

    # Step 2: Create TVAE model using proven ModelFactory pattern
    print(f"\n🏗️ Creating TVAE model using ModelFactory...")
    from src.models.model_factory import ModelFactory
    
    final_tvae_model = ModelFactory.create("tvae", random_state=42)
    print(f"✅ TVAE model created successfully")
    
    # Step 3: Train using .train() method (NOT .fit())
    print(f"\n🚀 Training TVAE model with optimized hyperparameters...")
    final_tvae_model.train(data, **final_tvae_params)
    print(f"✅ TVAE model training completed successfully!")
    
    # Step 4: Generate synthetic data
    synthetic_tvae_final = final_tvae_model.generate(len(data))
    print(f"✅ TVAE synthetic data generated: {synthetic_tvae_final.shape}")
    
    # Step 5: Quick evaluation using enhanced objective function (NO IMPORT - function in globals)
    if 'enhanced_objective_function_v2' in globals():
        tvae_final_score, tvae_similarity, tvae_accuracy = enhanced_objective_function_v2(
            real_data=data, synthetic_data=synthetic_tvae_final, target_column=TARGET_COLUMN
        )
        
        print(f"\n📊 TVAE Enhanced Objective Function v2 Results:")
        print(f"   • Final Combined Score: {tvae_final_score:.4f}")
        print(f"   • Statistical Similarity (60%): {tvae_similarity:.4f}")
        print(f"   • Classification Accuracy (40%): {tvae_accuracy:.4f}")
    else:
        print("⚠️ Enhanced objective function not available - using basic metrics")
        tvae_final_score = 0.5  # Fallback score
        tvae_similarity = 0.5
        tvae_accuracy = 0.5
    
    # Store results
    tvae_final_results = {
        'model_name': 'TVAE',
        'objective_score': tvae_final_score,
        'similarity_score': tvae_similarity,
        'accuracy_score': tvae_accuracy,
        'final_combined_score': tvae_final_score,
        'sections_completed': ['5.6.1'],
        'evaluation_method': 'section_5_1_pattern',
        'section_4_optimization': best_objective_score is not None,
        'best_section_4_score': best_objective_score,
        'optimized_params': final_tvae_params
    }
    
    print(f"\n✅ SECTION 5.6.1 - TVAE MODEL TRAINING COMPLETED!")
    print("-" * 80)

except Exception as e:
    print(f"❌ TVAE training failed: {str(e)}")
    import traceback
    traceback.print_exc()
    synthetic_tvae_final = None
    tvae_final_results = {'error': str(e), 'training_failed': True}

🏆 SECTION 5.6.1: BEST TVAE MODEL TRAINING
✅ Retrieved Section 4.6 TVAE optimization results
   • Best Trial: #1
   • Best Objective Score: 0.6493
   • Parameters: 8 hyperparameters

🏗️ Creating TVAE model using ModelFactory...
✅ TVAE model created successfully

🚀 Training TVAE model with optimized hyperparameters...
✅ TVAE model training completed successfully!
✅ TVAE synthetic data generated: (500, 6)
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4507
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8930
[CHART] Combined Score: 0.6276 (Similarity: 0.4507, Accuracy: 0.8930)

📊 TVAE Enhanced Objective Function v2 Results:
   • Final Combined Score: 0.6276
   • Statistical Similarity (60%): 0.4507
   • Classification Accuracy (40%): 0.8930

✅ SECTION 5.6.1 - TVAE MODEL TRAINING COMPLETED!
--------------------------------------------------------------------------------


### 5.2 Batch Process

This section outputs figures and graphics from models run in 5.1. The next chunk will output results for whatever subsections of 5.1 were run. I.e., if there's need to debug one method, there's no need to run all subsections of 5.1.

In [51]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# Following CHUNK_018 pattern with comprehensive file export to Section-5 directory
# ============================================================================

print("🔍 SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)
print("📋 Evaluating all available optimized models from Section 5.1.x")
print("📁 Exporting all tables and analysis to Section-5 directory")
print("🔄 Following Section 3 comprehensive evaluation pattern")
print()

# Ensure setup module function is available
from setup import evaluate_section5_optimized_models

# Use Section 5 batch evaluation function from setup.py
# Following exact same pattern as CHUNK_018 (Section 3) - comprehensive file export!
try:
    # Run batch evaluation with file export for all optimized models
    section5_batch_results = evaluate_section5_optimized_models(
        section_number=5,
        scope=globals(),  # Pass notebook scope to access synthetic data variables
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("✅ SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
    print("="*80)
    print(f"📊 Models processed: {section5_batch_results['models_processed']}")
    print(f"📁 Results exported to: {section5_batch_results['results_dir']}")
    
    # Show summary of all evaluations
    if 'evaluation_summaries' in section5_batch_results:
        print("\n📋 EVALUATION SUMMARIES:")
        print("-" * 40)
        for model_name, summary in section5_batch_results['evaluation_summaries'].items():
            print(f"🤖 {model_name}:")
            print(f"   📊 Synthetic samples: {summary.get('synthetic_samples', 'N/A')}")
            print(f"   📈 Overall score: {summary.get('overall_score', 'N/A')}")
            
    print("\n" + "="*80)
            
except Exception as e:
    print(f"❌ Section 5.2 batch evaluation failed: {e}")
    print(f"🔍 Error details: {type(e).__name__}")
    print()
    print("⚠️  Check that Section 5.1.x models completed successfully")

print("\n📈 Section 5.2 optimized model batch evaluation complete!")
print("🏁 Ready for final model comparison and production deployment!")

🔍 SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
📋 Evaluating all available optimized models from Section 5.1.x
📁 Exporting all tables and analysis to Section-5 directory
🔄 Following Section 3 comprehensive evaluation pattern

[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: breast-cancer-data
[INFO] Target column: diagnosis
[INFO] Variable pattern: final
[INFO] Found 6 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] GANerAid
   [OK] CopulaGAN
   [OK] TVAE

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/breast-cancer-data/2026-01-07/Section-5/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.743

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Over

In [ ]:
# Generate Section 5 README documentation
from src.utils.documentation import create_section5_readme

create_section5_readme(
    results_path=results_path,
    dataset_id=DATASET_IDENTIFIER,
    timestamp=SESSION_TIMESTAMP
)
